In [ ]:
!pip install python-docx
!pip install -U transformers
!pip install -q trl bert-score openai
!pip uninstall -y trl
!pip install trl==0.11.3
!pip install -q accelerate>=1.8.0
!pip install -q bitsandbytes>=0.46.1
!pip install -q datasets

import time
import random
import re
import pandas as pd
from docx import Document
import numpy as np
from datasets import Dataset
import os

import torch
torch.cuda.empty_cache()

import torch
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
from bert_score import BERTScorer
import openai
from datasets import Dataset
import numpy as np
import gc
from tqdm import tqdm

import bert_score
import transformers
import json
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

torch.cuda.empty_cache()
import gc
gc.collect()

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 111.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.8 MB/s eta 0:00:00
Found existing installation: trl 1.4.0
Uninstalling trl-1.4.0:
  Successfully uninstalled trl-1.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 23.4 MB/s eta 0:00:00
Mounted at /content/drive


Загружаем модель и токенизатор

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

Подключаем модель с openrouter по api

In [ ]:
USE_LLM_REWARD = True
OPENROUTER_API_KEY = ""
JUDGE_MODEL = "openai/gpt-oss-120b:free"

bertscorer = BERTScorer(lang="ru", rescale_with_baseline=False, device='cuda')

if USE_LLM_REWARD:
    openrouter_client = openai.OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY
    )
last_llm_score = 0.0

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ставим random_seed, чтобы ответ воспроизводился

In [ ]:
def set_random_seed(seed: int = 27):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True

set_random_seed()

Функции для LLM и reward

In [ ]:
llm_cache = {}

def get_llm_score(generated_text, source_text, topic):
    cache_key = (generated_text, source_text, topic)
    if cache_key in llm_cache:
        return llm_cache[cache_key]

    if not USE_LLM_REWARD:
        return None

    model = JUDGE_MODEL

    prompt = f"""Ты - социолог-эксперт в открытом кодировании интервью.
Тема интервью: {topic}
Текст интервью: {source_text}

Сгенерированные коды и цитаты (в формате <код>цитата</код>):
{generated_text}

Оцени результат по трём критериям (каждый от 0 до 1):
1. Релевантность - насколько код соответствует содержанию фрагмента интервью с учётом темы.
2. Когерентность - насколько формулировка кода грамматически корректна и стилистически приемлема.
3. Теоретический инсайт - степень соответствия кода концептуальным ожиданиям.

Ответ дай строго в формате: число, число, число (например: 0.85, 0.90, 0.75). Не пиши никаких пояснений."""

    max_retries = 5
    base_delay = 1.0
    max_delay = 30.0

    for attempt in range(max_retries):
        try:
            response = openrouter_client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=50
            )

            if response and hasattr(response, 'choices') and response.choices:
                content = response.choices[0].message.content
                if content:

                    numbers = re.findall(r"(\d+\.?\d*)", content)
                    if len(numbers) >= 3:
                        try:
                            scores = [float(n) for n in numbers[:3]]
                            mean_score = np.mean(scores)
                            mean_score = max(0.0, min(1.0, mean_score))
                            llm_cache[cache_key] = mean_score
                            return mean_score
                        except ValueError:
                            print(f"LLM warning: non-numeric numbers: {numbers}", flush=True)
                    else:
                        print(f"LLM warning: expected 3 numbers, got {len(numbers)}. Content: {content[:200]}", flush=True)
                else:
                    print("LLM warning: empty content", flush=True)
            else:
                print("LLM warning: invalid response structure", flush=True)

            return None

        except Exception as e:
            is_retryable = False
            if hasattr(e, 'status_code'):
                if e.status_code == 429 or (500 <= e.status_code < 600):
                    is_retryable = True
            elif '429' in str(e) or 'rate limit' in str(e).lower():
                is_retryable = True

            if not is_retryable or attempt == max_retries - 1:
                print(f"LLM error (final): {type(e).__name__}: {e}", flush=True)
                return None
            delay = min(base_delay * (2 ** attempt), max_delay)
            jitter = random.uniform(0, 0.1 * delay)
            wait_time = delay + jitter
            print(f"LLM error (retryable): {e}. Retrying in {wait_time:.2f}s... (attempt {attempt+1}/{max_retries})", flush=True)
            time.sleep(wait_time)

    return None

def compute_reward(generated, target, source_text, topic, use_llm=True):
    global last_llm_score
    _, _, bert_f1 = bertscorer.score([generated], [target])
    bert_reward = bert_f1.item()
    if use_llm:
        new_llm = get_llm_score(generated, source_text, topic)
        if new_llm is not None:
            last_llm_score = new_llm
        llm_reward = last_llm_score
    else:
        llm_reward = 0.0
    return bert_reward + llm_reward, bert_reward, llm_reward

In [ ]:
test_df = pd.read_csv('test_data.csv')
test_dataset = Dataset.from_pandas(test_df)

Загрузка тестового датасета

In [ ]:
def build_prompt_h1(example, tokenizer):
    """Формирует промт (query) как в исходном build_prompt, но без ответа"""
    instruction = f"""Ты — эксперт по анализу интервью. Твоя задача — выделить тематические коды в тексте интервью и привести соответствующие цитаты.

Пример текста:
Интервьюер: Так, все, все записи начались. Сначала я попрошу тебя рассказать какую-то основную информацию о себе – просто сколько тебе лет, чем занимаешься, где, с кем живешь, откуда ты вообще родом.
Информант: Мне 25. Родом отсюда, из Питера, никуда никогда не уезжала, всегда жила здесь. Вот. Занимаюсь сейчас фотографией. Э параллельно вообще училась на физика в [второй институт]. Вот. Но сейчас взяла академ отпуск по там по здоровью по причине. Вот. С сентября планирую вернуться. Живу с молодым человеком, снимаем квартиру. Вот. Как-то так.
Интервьюер: Угу. А можно еще немножко попросить тебя рассказать про своих родителей? Вообще, ну, может быть, у тебя еще есть братья, сестры. Вот про них, чем они занимаются, откуда они родом.
Информант: Ну, мы в общем-то все отсюда. У нас семья переехала с Украины. Вот. А мы, там мама моя она уже родилась здесь. Вот. То есть у нее детство так пятьдесят на пятьдесят прошло – то здесь, то в Украине. Вот. А мы с сестрой, с братом уже родились здесь. Сестра с братом у меня, правда, двоюродные, но как-то у нас так получилось, что мы все росли без отцов, поэтому мы вот очень-очень дружно между собой общаемся как родные. Вот. Брат сейчас, слава Богу, взялся за свою жизнь и занимается, ну работает, щас учится на, Господи, телесноориентированного психолога, что-то такое. Вот. У сестры очень много, очень много работы. У нее свой бизнес, она сдает виллы в аренду на Кипре, параллельно занимается бухгалтерией в компании. Вот. Как-то так. Это из основного.
Интервьюер: А они какого возраста?
Информант: Они гораздо старше меня. Сестре тридцать, если не соврать, тридцать шесть. Брату за 40 уже, то есть, ну, он на 18 лет меня старше. Вот. Ну вот они как-то вместе с бабушкой меня воспитывали, пока мама работала, поэтому у нас как-то такая странная, но дружная семья.
Интервьюер: А бабушка сейчас здесь?
Информант: Нет, бабушка умерла. Вот как раз завтра будет годовщина, четыре года. Вот. Ну, как бы ей уже много лет было просто. Вот.
Интервьюер: Поняла. Давай тогда еще немножко вернемся в прошлое. Можешь рассказать про свою школьную жизнь? Как она вообще проходила, что была за школа, какая там среда, что нравилось, что не нравилось?
Информант: Ээ школа абсолютно обычная. Сначала до четвертого класса училась в школе, которая вот прям перед домом. Вот [разговор с официантом] Но она как бы была такая не очень по отзывам людей со старших классов, да, у кого дети учились, и мама решила меня в другую перевести в военном городке. Вот. Там в принципе тоже школа обычная была, очень маленькая. Вот там я до девятого класса доучилась. Не знаю, то есть у меня была там типа своя небольшая компания, больше дружила с мальчиками. Вот. И одна подруга у меня там была, и все. Ну, как-то так отдельно от всех тусили, гуляли, всякое такое. Ничего особенного. Наверное, как и у всех школьные годы проходили. Вот. Училась я средне. Как бы умственные способности были хорошие, но я постоянно ругалась с учителями. Вот. У меня было какое-то обостренное чувство справедливости [смех] поэтому учителям это не шибко нравилось. Вот. Периодически из-за этого занижали оценки, но это скорее вот в 10, 11 класс когда я перевелась. Вот. А 10-ый, 11-ый я закончила языковую гимназию. Вот. Хотела заниматься языками, подумала, что вот она мне надо, пошла туда. В итоге в 10-м классе, как только я перевелась, я поняла, что мне интересна астрономия [смех] Занялась физикой.
Интервьюер: А как так получилось?
Информант: А я просто увидела на Ютубе видосик, там типа как будто камера отдаляется от Земли и там проходит через солнечную систему, там через галактику, так далее. Я такая: «Вау, это все настолько большое, интересно». Стала читать, смотреть, интересоваться, такая: «О, прикольно, я хочу этим заниматься». Ну и все, как-то так и пошло. Потом занялась физикой, чтобы подтянуть знания, стало интересно, поступила на физика. Вот. Щас должна была вообще, по идее, защищать диплом как раз по астрофизике. Вот. Ну, как бы через год пойду защищать.
Интервьюер: То есть 10-ый, 11-ый ты уже целенаправленно знала, да, чем ты будешь заниматься?
Информант: Да, я готовилась, очень, очень упорно занималась. У меня была куча репетиторов. Вот. Вообще много куда планировала поступать. В итоге в конце 11 класса я, ну, выгорела, я очень устала, потому что занималась очень упорно. Я хотела поступать потом через год. То есть дать себе год там поработать, отдохнуть от учебы, но мама сказала: «Нет, поступаешь. Куда поступишь, туда поступишь». Поступила в [первый институт]. Жалею, что поступила, оно мне не надо было, тогда, действительно, мне нужен был перерыв. Вот потом, когда я уже после [первый институт] пошла в [второй институт], ну, [первый институт] я не закончила. Вот. Вот тогда прям вот. Мне был 21 год, оно было как раз к месту, вовремя, и университет гораздо, как сказать, дружелюбно настроенный к студентам нежели в [первый институт] это было. Вот. То есть как-то у нас хорошая коммуникация и с преподавателями, и с деканатом, все там могут если что войти в положение, со всеми можно договориться, если там где-то что-то упустил, тебе помогут подтянуть. В [первый институт] такого не было. Ну, и как бы у нас одна группа в [второй институт], а в [первый институт] поток 120 человек, там каждому помочь тоже, ну, нет. Вот. Поэтому я сейчас в принципе университетом очень довольна. Вот. Я уже отошла от темы.
Интервьюер: Нет, нет, мы как раз дальше поговорим про в принципе как складывается жизнь как раз-таки именно после школы. Вот как бы понятно, что вуз получается ты выбирала целенаправленно, да. Ну, в принципе тоже что нравилось в этот период, что не нравилось, как он проходил?
Информант: Ну, в вуз изначально вообще я очень хотела ээ в какой-нибудь другой город поступить. Вот. Но мама сразу резко сказала: «Нет, на это у нас нет денег, я не смогу тебя содержать где-то там». Вот. А я понимала, что если я поступлю там в тот же [московский вуз], о котором я мечтала, то, ну, как бы работать параллельно я не смогу, надо будет полностью отдаваться учебе. Тем более что я изначально не, как сказать, у меня нет предрасположенности к физике и к математике, я, скорее, изначально больше как гуманитарного склада ума. Вот. Мне нужно было бы там в 10 раз сильнее работать, чем другим ребятам, чтобы нагнать, чтобы успевать за ними. Вот. Поэтому пришлось все-таки остановиться на Питере. Изначально я думала, что это будет [вуз в СПб], и как бы нацеливалась туда на матфак. Вот. Но вот как раз когда я решила, что нет, все, я не хочу сейчас никуда поступать, я больше не буду стараться. Последние месяцы перед ЕГЭ я полностью забила на ЕГЭ. На экзамен пришла просто на отвали, лишь бы вот от меня все отвязались, особо не старалась его сдавать. Ну, как бы в итоге и баллы такие получились, потому что особо не старалась. Я там, по-моему, во вторую часть даже не залазила особо, так пролистала, там че-то одну задачу сразу пришло в голову как решить, записала, и все, остальное даже не пыталась. Вот. Ну, в итоге как бы я такая: «Вот, низкие баллы, я никуда не пройду». Мама сказала, что нет, ты должна хотя бы попытаться подать. Я подала в [первый институт]. В один вуз на одно направление. И я прошла последней во второй волне на бюджет. Мама такая сразу: «Ну, раз прошла, надо идти. Ну, повезло. Ну, надо». Я думаю: «Ладно, пойду, посмотрю, попробую. Не понравится – всегда можно забрать документы и уйти». Но в итоге сразу забрать документы и уйти мне не дали. Вот. Сказали: «Ну, нет. Поступила – попробуй, поучись». Ну, я попробовала, поучилась полтора года получается где-то. Потом все. Ну, там уже у мамы были проблемы с деньгами, ей нужно было, чтобы я помогала работать. Вот. Зарабатывать денежки, потому что бабушка у нас инвалид второй группы была, мама уже предпенсионного возраста, ей было сложно найти работу, а ее тогда сократили. Ну, как бы кому работать? Вот. Поэтому тогда уже от меня отстали, сказали: «Ладно, все, отчисляйся. Давай. Пока работай. Пока надо работать». Вот. Как-то так. То есть мне, наверное, тогда, когда я поступала, в 18 лет, не столько была интересна учеба. То есть вот этот вот этап, когда мне было очень интересно учиться, я пережила в школе, когда я готовилась к ЕГЭ. И когда я выгорела, я решила, что я вот жизнь, я хочу работать, я хочу там тусить, развлекаться, знакомиться с людьми, там, не знаю, ходить на концерты, но учебу я хочу попозже. То есть мне хотелось как-то вот познать эту жизнь, так сказать. Вот. Поэтому учеба меня как бы, наверное, не очень сильно интересовала, именно вот университетский формат учебы. Вот. Поэтому в университете по большей части я там обзавелась друзьями, мы ходили тусили, гуляли. Вот. Там всякие студенческие организации, я в них там очень активно участвовала. Ну, как бы вот так вот. Вот. А вот когда в [второй институт] поступила, там я уже четко понимала, что я хочу учиться, чем я хочу заниматься, что оно мне надо, и я в общем туда пошла учиться и учусь. Вот. Вот так вот.
[сделали небольшой перерыв]
Интервьюер: Угу. Смотри, ты сказала вот про этот опыт, когда ты, получается, перепоступала. Скажи, пожалуйста, ну, я так понимаю, что работать – это все-таки была такая, наверное, необходимость, но и твое как бы нежелание учиться в [первый институт], как-то все совпало?
Информант: Да.
Интервьюер: А вот когда, получается, ну, вот как пришло понимание, что ты хочешь еще раз поступать? И как бы как вот это соединилось, что работа и учеба? Ты совмещала или ты перестала работать?
Информант:  Изначально я даже когда из [первый институт] уходила, я понимала, что я рано или поздно все равно пойду учиться, что я все равно хочу, как сказать, получить диплом физика, вот это вот написать дипломную работу по астрофизике. То есть даже если я не буду работать дальше в этой стезе, все равно мне хотелось получить этот опыт именно в этой области. Вот. Поэтому понимала, что куда-то я все равно поступлю, что-то с этим сделаю. Может быть, даже не именно в астрофизике, мне, скорее, хотелось просто куда-то в науку. То есть у меня были мысли поступать на биологический факультет, там нейрофизиологией заняться, что-нибудь такое. Вещи, которые я вообще не понимаю, но мне было очень интересно. Вот. Меня как-то всегда в какие-то такие штуки тянуло, и вот думала: «Почему нет?» В итоге я просто не стала пересдавать ЕГЭ и пошла со старыми баллами ЕГЭ на физику. Вот. Изначально когда поступала, я работала. Я работала в [визовый центр], где вот мы с [общая подруга] познакомились. Вот. Мне в целом там работа нравилась, как бы все устраивало, все было хорошо. Вот. Но там полноценный рабочий день. И когда я поступила на первый курс в [второй институт], у меня очень много перезачетов было с [первый институт]. Взяла справку, у меня предметы, да, все типа, преподы сказали: «Все ок, перезачитываем». И у меня было мало пар, я могла спокойно совмещать с работой. Вот. Но потом деканат меня прижал, они сказали: «Нет, так не пойдет» [смех] Типа: «Ты должна ходить на все пары. Перезачет, не перезачет – ты все равно должна ходить на пары. Я говорю: «Ну как? Мне надо работать». Они такие: «Ну, ты поступала на очку». Типа: «Ты должна была понимать, что к чему». Вот. Мне пришлось уволиться. Мы договорились с мамой, что, ну, как бы она на тот момент уже работала, у нее уже было все более-менее хорошо. Вот. Договорились, что как бы да, я попробую найти какую-то подработку, но все-таки в основном она будет работать, меня содержать, пока я учусь. Вот. Ну, в общем-то так и получилось. Правда, подработку я не скоро смогла найти, потому что корона началась. Вот. А после короны уже нашла подработку и параллельно работала в планетарии. Там были экскурсии, там лаборатория есть, и вот я по лаборатории экскурсии водила. Вот. Как-то так.
Интервьюер: Угу. Если вот говорить про именно какую-то рабочую, да, трудовую жизнь, то расскажи, наверное, подробнее вообще про свои работы. Какие они были, что тебе на них нравилось, что не нравилось? Ну, и вот что в принципе тебе хочется в трудовой сфере?
Информант: Мм работ у меня было очень много. В первый раз я пошла, ну, это так, не работа была, это такая двухдневная подработка была. Меня вписали в, Господи, в массовку в фильме сняться. Мне было 14 лет. Вот. Это оказался адский труд, потому что мы снимали конец ноября в середине июля. Вот. Было очень жарко, там че-то на улице плюс 30, а мы типа в тулупах, там шерстяных шарфах, шапках, всякое такое. Не помню, сколько заплатили, но на данный момент мне кажется очень мало за такое дело. Вот. Потом уже полноценно я устроилась в первый раз работать в 16 лет. Когда были каникулы, приехала к сестре от мамы. Вот. Рядом с ее домом устроилась работать поваром в [кафе]. Вот. В целом, ну, как бы на тот момент я очень радовалась, что вот я работаю, наконец-то я могу там себе что-то купить, что мне хочется. На первую зарплату я купила себе походный рюкзак, потому что мне не хотелось ходить со старым походным рюкзаком моей мамы в походы. Вот. Ну, и так в целом немножечко там сестре помогала, там покушать покупала, гуляла, вот всякое такое. Вот. Потом уже я устроилась работать после школы, как раз перед поступлением в [первый институт]. Я устроилась работать в, Господи, как это назвать, ээ типа как, ну, вообще это как анти-кафе было, но параллельно как площадка для мероприятий. Вот. Это называлось [анти-кафе], не знаю, может, посещала когда-нибудь. Сейчас он уже закрыт давно. Вот. Очень классное место было, мы там все как эта, как такая тесная семейка стали, все, кто там работал. Вот. Я там начала встречаться с молодым человеком, и у нас как-то дом, работа, мы везде вместе были. Вот. И в целом я, ну, нормально, вот эту работу я нормально совмещала с учебой. Ну, как бы не скажу, что прям хорошо я училась параллельно с этим делом, потому что все-таки как-то работать мне нравилось больше, чем учиться на тот момент. Вот. Плюс на работе у нас проходили постоянно какие-то мероприятия, концерты, какие-нибудь мастер-классы, и я помогала это все, ну, расставлять, организовывать, связываться с теми, кто проводит мастер-классы, там с музыкантами. Вот. И, конечно, это интересно было. Интереснее, чем учеба на тот момент. Вот. Зарабатывали мы там совсем немного, но как бы вот этот вот интерес, который там был, он как-то все восполнял. Вот. Потом, когда вот [анти-кафе] закрылся аа щас вспомню, что же там уже дальше было [смех] Уже плохо помню, куда я дальше устроилась работать. А, ну, я училась, да, вот дальше в [первый институт], и в какой-то момент, когда начало не хватать денег, я там сидела, грубо говоря, у мамы на шее сначала некоторое время, где-то полгода, наверное. Потом меня сестра устроила к своему товарищу в мебельный салон мебель продавать. Вот. Чисто на новогодние праздники, что-то такое. Вот. Я там попродавала, все, типа там вот буквально новогодние праздники, и все. Вот. И после этого, по-моему, я как раз-таки ушла из [первый институт], потому что, ну, вот нужны были деньги. Вот. Так, уже дальше не помню конкретно по хронологии. Я работала в [магазин украшений] где-то год, продавала украшения, продавец-консультант. Вот. Потом работала продавцом-консультантом в каком-то магазине одежды, но совсем недолго. Там чисто вот пока я искала новую работу. Вот. Работала вот как раз в [визовый центр]. Сначала туда на телефон устроилась, потом я уговорила, чтобы меня перевели в визового эксперта. Там все изучила, все выучила. Вот. Все экзамены, типа того у них сдала, на вопросы ответила, и меня сделали визовым экспертом. Вот. И вот после этого уже устроилась, когда училась в [второй институт], в планетарий. Вот. Там экскурсии проводила. По-моему, все. Да, как-то так.
Интервьюер: Поняла. А что вообще для тебя важно в работе, да, какие у тебя трудовые ценности, если можно так сказать?
Информант: Ну, на данный момент мне очень нравится, что я, как сказать, у меня нет какой-то отчетности перед кем-то, да, что я в конкретное время должна в конкретном месте появиться. Вот. Сейчас для меня это очень ценно, потому что, ну, в связи с тем, почему я взяла академ, как бы у меня есть сложности иногда с выходом из дома. Вот. А в остальном, наверное, для меня всегда важно было, чтобы мне было в целом комфортно на работе, да. То есть обязанности, которые я выполняю, были мне интересны. При этом в какой-то момент я поняла, что мне жутко не нравится работать с большим количеством народу. То есть мне комфортно если работать с людьми, то один на один. Либо вообще работать не с людьми, а что-то другое делать. Вот. Поэтому я вот сейчас когда я снимаю, мне вообще вполне себе прекрасно. То есть я устанавливаю контакт с заказчиком, стараюсь заранее встретиться перед съемкой, с ними как-то познакомиться, поговорить. Вот. Наладить общение, чтобы и они себя более комфортно чувствовали, и я примерно их понимала уже. Вот и дальше, когда я снимаю, даже если там, ну, вот последняя съемка у меня была, там было пять человек, но это вся семья, я их как-то воспринимала как вот единое целое, и как бы было достаточно комфортно. Вот. Когда одного, двух людей снимаешь, так вообще прекрасно. Так что вот так.
Интервьюер: Угу. А если говорить в терминах самостоятельности, взрослости, да, то считаешь ли ты себя сейчас самостоятельным, взрослым человеком?
Информант: К сожалению, нет. Все равно есть такой момент. У меня молодой человек достаточно хорошо зарабатывает. И в принципе когда вот мы стали с ним встречаться, я работала в планетарии тогда, зарплата, ну, соответственно, небольшая была, потому что работала неполный рабочий день, потому что надо было учиться. И на тот момент там были очень сложные всякие моменты именно в самой работе из-за того, что там задерживали зарплаты, чего-то начинали урезать, кого-то начинали увольнять. Я понимала кхм-кхм Вот. Я понимала, что оттуда надо уходить, куда пока не понимала, куда я еще смогу пойти, чтобы параллельно учиться спокойно, это не мешало учебе. Вот. И вот мы тогда уже жили вместе с молодым человеком, и он сказал типа: «Увольняйся, моей зарплаты хватит на нас двоих. Пока учишься – учись. Не переживай из-за денег». Вот. Я сначала как-то мне некомфортно было, потому что я не привыкла у кого-то сидеть на шее. Вот. Но понимала, что для учебы так, действительно, будет лучше. Вот. И уволилась. Собственно, некоторое время я вообще не работала. То есть я пыталась там что-то где-то подрабатывать. Я вела английский разговорный клуб, особо с этого денег не поимела. То есть как бы это скорее больше как увлечение получилось, чем заработок. Вот. И потом вот когда появилась возможность купить фотоаппарат, купили фотоаппарат, я начала вот этим заниматься. И уже как бы более-менее сейчас я начинаю зарабатывать, но все равно большая часть финансов, которые уходят на меня, уходят со стороны моего молодого человека. Поэтому пока, к сожалению, про самостоятельность, ну, не могу говорить. Вот. Очень хочется, но пока, пока нет. Вот так.
Интервьюер: Угу. А вот когда, ну, как бы, ну, получается, у тебя не было, да, такого периода, когда вот ты жила как-то одна? То есть с мамой, потом с сестрой, да, потом с молодым человеком?
Информант: Ну, у меня был небольшой период. Опять-таки я тоже жила с молодым человеком, с предыдущим, с бывшим. Вот. Ну, там у нас скорее получилась такая ситуация, что он был младше, и он такой типа на чилле, на расслабоне, ниче не хочу делать, то хочу, то не хочу, и я тянула, собственно, нас двоих. На тот момент вот я как раз работала в [магазин украшений] и потом в [визовый центр], в визовом центре. Вот. Ну, то есть опять-таки я не жила одна. Вот. Но на тот момент это, наверное, был пик моей самостоятельности, когда я содержала и себя, и парня. Вот. При этом еще умудрялась откладывать деньги, умудрилась поехать в Европу в отпуск на две недели. То есть как бы в целом я тогда была очень довольна собой. У меня было еще, а, у меня было две работы параллельно. Помимо [визовый центр] я тогда еще работала а-ля как личный ассистент своей сестры вот по поводу ее бизнеса на Кипре. Вот. Но это сотрудничество у нас недолго продолжилось, потому что все-таки сестринские отношения страдали. Я это быстро поняла, и в общем-то буквально несколько месяцев я на нее проработала, и мы перестали работать вместе. Так что вот так. А сейчас, да, я как такой персональный нахлебник моего парня [смех] Но вроде он не парится. Я немножечко переживаю по этому поводу, конечно, но со временем подпривыкла. Вот. Не то, чтобы у меня сильно какие-то большие запросы, поэтому вроде ему нормально, а я постепенно стараюсь выйти к самостоятельности. Вот. Успокаиваю себя тем, что я все равно не сижу на месте. Просто, не просто сижу живу за его счет, а что-то пытаюсь сделать, как-то развиваюсь и так далее. Вот. Вот так.
Интервьюер: А вот в этом периоде, да, ну, все равно, да, когда ты там уезжаешь от родителей, да, ну, все равно как-то жизнь меняется, да. Вот что стало легче, что стало сложнее?
Информант: Однозначно легче стало общение с мамой. Гораздо. Потому что у нас с ней есть такой момент, что все, чем вот я в себе недовольна, то же самое есть в ней, и она в себе этим недовольна. И когда мы видим вот эти вот свои негативные черты в другом человеке, то злимся на него еще больше за это. И у нас часто были какие-то моменты, что мы не могли даже, вот мы находимся в одной квартире, просто я в своей комнате сижу, она в своей, и мы не можем друг с другом пересекаться. А сейчас когда мы живем отдельно, у нас прям прекрасные взаимоотношения, мы чуть ли не каждый день созваниваемся, по 2 часа можем говорить, все на свете обсуждать. Вот. И соскучиться успеваем, всякое такое. Вот. Мне стало гораздо проще говорить, что я ее люблю, обнимать ее, целовать. Вот. То есть в этом плане, да, стало легче. Ээ в остальном я бы не сказала, что меня как-то сильно ограничивали, пока я жила с мамой. Вот. То есть там по времени где я нахожусь, что я делаю, никто особо не заморачивался, даже когда я подростком была. Вот. То есть мне достаточно много свободы давали. Вот. Поэтому не знаю. В целом, наверное, мне не очень нравилось еще жить в своей квартире. Квартира была моей бабушки. Сейчас в ней живет моя мама. Вот. Ну, в общем-то мне скорее район не нравился. Вот. Мне очень хотелось оттуда выбраться. И вот в плане того, что я оттуда выбралась, я, наверное, очень рада.
Интервьюер: А что там было не очень? Типа безопасность?
Информант: До метро ехать неудобно. Да, сам по себе район такой, не знаю, как сказать. Мне не нравилось, что я постоянно выхожу на улицу и кого-то из знакомых встречаю с детства там и так далее. Вот. Мне очень этого не хотелось. Все сразу начинают расспрашивать чего, где, кто, как. Вот. Ну, как-то не знаю. В целом мне там вот некомфортно было. То есть чтобы выехать куда-то в центр, с кем-то встретиться, это надо полтора часа потратить. Вот. И иногда сидишь и думаешь просто: «Да, ну, его нафиг. Никуда не поеду». Либо наоборот если выезжаешь, то ты сразу себя готовишь к тому, что после там, если выезжаешь на учебу, то после этого там еще должен пойти, не знаю, в библиотеку позаниматься, значит ноут с собой, там с кем-то с друзьями встретиться, там может одежду или косметику переодеться, там накраситься с собой. И вечно тащишь с собой такие чемоданы.
Интервьюер: Да, да, я прекрасно тебя понимаю, потому что я вот сегодня именно в таком [смех]
Информант: Вот да. Спина болела перманентно [смех] Поэтому было дико неудобно. А щас когда я вот на [место жительства] переехала, университет у меня на [место второго института], было безумно удобно, когда у меня там перерыв между парами, окно, я могу заехать домой, покушать и вернуться обратно в универ.
Интервьюер: Вообще чудесно.
Информант: Да. Вот. То есть, наверное, вот в этом плане.
Интервьюер: А на какие этапы, периоды ты вообще могла бы свою жизнь разделить?
Информант: Мм вопрос сложный, наверное. Я на самом деле несколько раз съезжала оттуда, вот с [первое место жительства] от мамы, и возвращалась потом туда. Получается, если брать с переездом к сестре, наверное, раз, два, три, четырежды. Четырежды я съезжала, вот трижды пока возвращалась, надеюсь больше не вернусь [смех] Вот. Поэтому, наверное, у меня вот как-то жизнь делится между этими периодами, когда я съезжала и возвращалась. Вот. Когда еще бабушка была жива, с ней мы очень тяжело уживались. То есть у нас нормальные отношения были только когда я с ней не жила вместе. Вот. Поэтому мне было очень комфортно жить отдельно от них. Когда бабушки уже не стало в принципе было не настолько заметна разница. Вот. Потому что с мамой, ну, все равно у нас более-менее хорошие отношения были, даже когда мы жили вместе. Вот. Но все равно то есть было вот это вот изменение. Вот. Поэтому, наверное, да, вот периоды когда переезжала откуда-то куда-то. Вот. Аа в остальном, наверное, вот разделить можно по учебе в универе. Когда в [первый институт] поучилась, ушла, поработала, в [второй институт] пришла, поучилась, сейчас пока временно перестала. Вот. Как-то так.
Интервьюер: А вот какие-то ключевые, да, моменты, события, тоже периоды в жизни можешь назвать?
Информант: Ну, выпуск со школы, когда я устроилась работать, поступила в [первый институт]. Это так или иначе. Поездка в Европу. Очень, как сказать, я до этого никогда особо за границу не выезжала, в Украину разве что, в Египет, такой пляжный отдых. Вот. А тут я сама в одиночестве поехала по Европе, по разным странам, познакомилась с большим количеством людей. У меня там появился молодой человек из-за границы. Вот. Это, как сказать, какой-то такой новый международный опыт, которого у меня никогда не было. Когда смотришь, что на самом деле, ну, путешествовать – оно гораздо проще. И люди везде такие же люди, да, там говорят на другом языке, может немного другой менталитет. Вот. Но до этого мне все это казалось каким-то таким сильно недоступным. Вот. А потом, когда я там, ну, путешествие я хреново продумала, честно говоря, я там встревала в несколько неловких ситуаций. Вот. Но когда я из них выпутывалась, я такая: «Ну, я могу все в этой жизни». Вот. Наверное, поэтому мне было очень комфортно. Я в этой поездке посетила место, о котором очень мечтала. Это ЦЕРН, там Большой адронный коллайдер находится. Вот. Для меня это прям мечта громадная была. Не знаю, что еще, может быть NASA я бы еще хотела посетить как БАК. Вот. Поэтому вот, наверное, да, тогда очень, очень важный момент для меня был. Вот. В целом вот не знаю, наверное, сейчас в какой-то степени очень важный период проходит, потому что вот пришлось взять академ ээ из-за того, что вот мне поставили. У меня так очень много переосмысления происходит. Я очень много сижу наедине с собой, со своими мыслями, хожу к психологу. Вот. Отношения с молодым человеком меняются, вроде как, дай Бог, идут к свадьбе [смех] Не знаю, чем это закончится. Вот. Сейчас я наконец-то занимаюсь фотографией. Я этим хотела заняться, еще учась в школе. Вот. То есть у меня всегда было представление, что, ну, я буду учиться, а параллельно буду снимать, вот это будет мой доход. Но проблема была в том, что накопить на фотоаппарат нормальный я никак не могла. Вот. А тут у нас удалось, получилось. Вот. И я очень этому рада, что вот у меня, наконец, есть возможность заниматься тем, что мне хочется, и вроде как то, что у меня получается. Вот. Как-то так.
Интервьюер: Ну, сейчас еще хочу подробнее расспросить про последние два-три года, да. Ну, и твои личные какие-то события, и события в мире, они так или иначе влияют на нашу жизнь. Вот какие события, изменения в тебе, в твоей жизни произошли именно за этот период? Можешь как-то хронологически, если тебе удобно. Если нет, то просто какие-то.
Информант: Ну, вот если корона. Очень сложно я ее восприняла морально, потому что у меня прям все, все было четко распланировано в общем. В плане тех же путешествий. То есть я только начала куда-то выезжать. У меня появился друг из Италии, который жил в Иордании, который ждал меня к себе в гости. У меня появился молодой человек, который жил в [страна, где жил прошлый молодой человек], и нам нужно было тоже как-то видеться, коммуницировать. Вот. И у меня был такой четкий план. Я должна была, [дата дня рождения] у меня день рождения, я должна была на него поехать в Таллин. После этого 26 марта, как щас помню, я должна была поехать к другу в Иорданию. У меня уже все вот, практически были куплены билеты, он сказал: «Подожди, у меня вопросы по работе. Я не уверен, что именно в этот день смогу тебя встретить. Если что я тебе сам прям дата в дату куплю билет». Вот. Если бы не это, у меня просто прогорели бы билеты, которые я планировала покупать. Вот. Потом после этого я должна была поехать, а, вру, ко мне должен был в апреле приехать вот молодой человек. После этого в мае я должна была к нему приехать. То есть у нас прям все, планы были глобальными.
Интервьюер: Грандиозные.
Информант: Вот. И все эти планы рухнули, потому что ровно как раз в 20-х числах марта началась корона, и как бы все, никуда не выедешь. Вот. Поэтому в этом плане, да, мне пришлось сидеть дома. На тот момент меня это как-то, мне очень сложно было, не знаю. Сейчас мне, наоборот, очень комфортно сидеть дома, чтобы меня лишний раз никто не трогал. А тогда мне казалось, что я такой дикий экстраверт, и я не могу сидеть дома, это вообще невозможно. Меня постоянно порывало куда-то пойти. У меня тоже вот в том же районе жила на тот момент лучшая подруга. Вот. И мы, собственно, постоянно с ней вот выходили, гуляли у нас по райончику. Вот. То есть никуда далеко выезжать не приходилось, и там у нее дома сидели. То есть как-то так. Это в принципе было единственное развлечение. Ну, параллельно учеба дистанционная была, конечно. Вот. А потом именно, назовем это, как называют в СМИ, специальная военная операция когда началась, задело меня очень сильно. Потому что, как я уже упоминала, у меня корни с Украины. Ну, я как бы сама там была один раз всего, месяца два провела. Вот. Но все равно как-то. У меня в детстве бабушка со мной на украинском разговаривала. Вот. Сейчас я уже ничего не помню на этом языке, только, ну, понять могу, наверное, как любой русский, вот в таком формате, а сказать ничего толком не могу. Вот. А в детстве понимала, говорила свободно, все прекрасно. Вот. И как-то мне бабушка прививала любовь к этой культуре, именно что вот всегда говорила: «Ты украинка. Нет, ты родилась в России, но ты украинка». У меня это как-то на подсознании отложилось. И тут когда начинается [СВО] между как бы странами, которые вот я в одной родилась, выросла, а другая мне всю жизнь говорили, что я украинка, я из этой страны. Вот. Ну, это тяжело ударяет, как бы типа ты такой не понимаешь, а как, а что. Вот. А когда началась мобилизация, у меня уехала сестра за границу вместе с. Сначала уехала ее муж Вот. Потом уехала сестра вместе с племянницей. Я очень переживала, потому что мы прямо супер близко общаемся, постоянно видимся. Вот. А тут все, как бы они за границей, неизвестно когда я их увижу, увижу ли вообще. Вот. Плюс из-за мобилизации, у меня с молодым человеком разные взгляды на эту тему, и он тогда сказал: «Позовут служить – я пойду». Вот. Я говорю: «Куда ты пойдешь, дурачок? У меня с той стороны дядя служит». Он из Одессы, и он военный. Вот. Я говорю: «Ну, ты представь, да, как бы вероятность, может, невелика, но представь такую ситуацию: ты пойдешь, я не знаю, дядя мой тебя убьет там на фронте, или ты его убьешь». Я говорю: «Ну, как бы зачем на это добровольно соглашаться»? Ему была компанией предоставлена возможность переехать куда-нибудь, спрятаться от мобилизации. Вот. На что он как раз-таки сказал: «Нет». Типа: «Позовут – я пойду». Вот. Мы из-за этого очень много ссорились, спорили. Я думала уже все, отношениям конец, ничего не выйдет. Вот. И в общем-то из-за этого, из-за этого в том числе, начались очень сильные проблемы у меня с головешкой. Вот то, что я говорила, что мне там поставили заболевание, мне поставили депрессивное невротическое расстройство. Вот. Я сейчас на антидепрессантах сижу, и первое время было, ну, супер тяжко. То есть как-то вот оно вроде бы такие вещи, которые у всех происходили, то есть не сказать, что у меня там какое-то громадное горе в жизни произошло, но как-то оно все так наслаивалось, наслаивалось, видимо, ну, просто как бы ментальное мое, как сказать, психологи психологическое что-то там внутри короче не выдержало и поломалась. Вот. И было, ну, прямо тяжелое состояние. Типа вот я просто не видела смысла что-то делать, вставать с утра там, я не находила сил почистить зубы банально. То есть, не знаю, в туалет я хотела до последнего, то есть когда уже, ну, совсем все разрывает, тогда я пойду в туалет, а до этого я не встану с кровати. И там и суицидальные мысли были, и такие практически уже доходящие до попыток. То есть это на самом деле отразилось очень-очень плохо. Вот. Я вот сейчас более-менее последний месяц только прихожу в себя. Вот. Поэтому в общем-то и взяла академ, потому что, ну, как бы я просто не в состоянии была куда-то выходить. У меня как только я выходила на улицу, случались панические атаки, когда я оказывалась с большим количеством людей. Вот. И доехать до универа просто было нереально. То есть я все, я сидела дома, заказывала доставку продуктов, и больше ничего. Вот. Поэтому вот так вот. Как бы не знаю, если бы не антидепрессанты, че бы я делала. Вот. Щас вроде стараюсь как-то поспокойнее ко всему этому относиться. Ну, то есть, как бы я не могу ни на что повлиять. У меня нету позиции а-ля там кто прав, кто виноват. Меня вообще не волнует, кто прав, кто виноват. У меня просто есть как-то свое мировосприятие, что, ну, [СВО] – это плохо. Нас всю жизнь так воспитывали. Нападать на кого-то – это плохо. Все. Остальное как бы я ничего не знаю, никого не хочу там говорить, что кто-то плохо поступает и так далее. Ну, как бы так. Вот. Просто очень жду, когда это все закончится. Вот. И стараюсь поменьше читать новости, потому что они сильно триггерят, потом опять лежишь весь никакущий абсолютно. Вот.
Интервьюер: А, ну, вот сейчас считаешь, что чувствуешь себя, ну, как бы если можно сказать лучше именно благодаря терапии?
Информант: Я думаю, да. Ну, по антидепрессантам я прям четко ощущала, то есть когда я принимала, когда мне корректировали дозу препарата, я ощущала. То есть первое время меня как-то штормило в разные стороны – то мне очень хорошо, то мне очень плохо. Вот. Потом когда мы подобрали конкретный препарат, конкретную дозу, постепенно я ощущала, что я вроде как прихожу в норму. Вот. То есть в этом плане, да. Плюс, наверное, очень хорошо, что у меня сейчас была возможность, то есть дать себе, собственно говоря, посидеть дома. То есть не работать, не учиться какое-то время. Вот. Это вот большое спасибо моему парню за это. Потому что, наверное, если бы мне приходилось работать в таком состоянии, ну, я не знаю, чем бы это закончилось. Я, ну, как бы, ну, как минимум, меня бы уволили, потому что я была абсолютно не работоспособным человеком в это время. Вот. Так что так вот.
Интервьюер: Ну, вот ты говорила про то, что, ну, как бы с парнем случился разлад из-за разных позиций, да. Но были ли за этот период еще какие-то ситуации, когда приходилось, может быть, делать какой-то выбор? А-ля там, может быть, думали о переезде, что-нибудь такое?
Информант: Ну, я в принципе думала о переезде так или иначе, потому что, ну, у меня специальность такая, что в России за нее, к сожалению, много не платят, ученые-теоретики особенно. Вот. А я все-таки хотела пробовать себя в этой стезе, именно работать. Я думала поступать на магистратуру в Германию. Вот. Потому что там эта область как раз-таки очень востребована и очень хорошо развивается. Вот. Ээ сейчас уже не знаю, я просто уже не хочу как-то далеко планировать. Вот. А тогда, когда у меня уехала сестра, она планировала меня забрать с собой. То есть у нас был план, что вот я закончу универ в этом году, и все, уеду к ней, а там дальше разберемся, чего, как. То есть, может быть, я там бы поступила, может, еще что-то. Ну, в итоге не пришлось, сестре пришлось вернуться сюда, потому что с документами были проблемы. Вот. Они сейчас здесь. Поэтому да, мысли уехать были, но парень у меня все равно как-то прям ни в какую не хотел. То есть он такой очень ярый патриот своей страны. Он никуда не хочет уезжать. Вот. А я как-то наоборот, мне всегда было интересно пожить где-то в другом месте. То есть я очень люблю свою страну, но при этом я как-то отношу себя больше как гражданин мира. Мне вот как бы, не знаю, у меня нет в голове какого-то такого представления жесткого о границах, да, о территории и так далее. То есть типа почему бы нет? Вот. Главное просто все организовать правильно. Почему бы не пожить в другом месте? Не вижу в этом ничего плохого. Не понравится – вернешься обратно или там уедешь в другое место. Почему бы, ну, почему не попробовать? Вот. А сейчас вот просто в какой-то момент, когда когда как раз-таки случилась депрессия, мой парень меня очень, очень здорово поддержал. То есть я от него вообще не ожидала. У него достаточно, как мне кажется, низкий эмоциональный интеллект. Он, ну, он такой типичный мужик, такой русский мужик, который эмоций не проявляет. А тут он прям, он был рядом, он, как сказать, адекватно все воспринимал, как бы входил в положение, выслушивал, успокаивал меня, находил нужный слова. Вот. И мне в какой-то момент показалось, что мне важнее находиться с ним и получать от него вот эту поддержку, быть рядом с этим человеком, чем куда-то переехать. Вот. Пока что для меня в принципе оно так и есть. Не знаю, потом посмотрим. Может, может, что-то изменится. Вот. Мы сейчас ходим к семейному психологу [смех]
Интервьюер: Ого.
Информант: Да. Вот. Пытаемся все вопросы как-то выровнять, наладить, потому что все равно многое остается. Вот. Как-то так.
Интервьюер: Поняла. Если вот не касаться, да, этих общих, общемировых, так скажем, ситуаций, то, может быть, тоже за последние два-три года вот лично у тебя какие-то изменения, события, которые как-то что-то поменяли, изменили, повлияли на тебя?
Информант: Мм даже не знаю. Наверное, ну, вот отношения. Мы два года встречаемся. Когда они начались, очень, гораздо проще стало, потому что у меня как раз до этого были очень тяжелые вот эти вот отношения на расстоянии. Вот. Которые уже надо было заканчивать, а я все никак не могла закончить. Уже и парень сам говорил, что надо. Вот. А я все как-то: «Нет, нет, нет. Я все сделаю, я к тебе приеду. Пожалуйста, давай не расставаться». Вот. И вот когда появился [молодой человек], я как будто выдохнула. То есть мне наконец-то стало спокойно и безопасно. Вот. И с этим ощущением спокойствия, безопасности жить что ли легче стало, не знаю. В остальном, наверное, нет. Ну, вот только то, что я описала именно с общемировыми, да, ситуациями. Вот. В остальном я как бы вот за последние два-три года я училась, подрабатывала, то есть как-то все достаточно ровно было. Вот. Так в остальном.
Интервьюер: А есть ли в этом периоде тоже какие-то ситуации, события, которые бы тебе хотелось как-то переделать, или вообще чтобы их не происходило?
Информант: Ну, у меня год назад у мамы инсульт случился. Вот. Хотелось бы, чтобы такого не было. А так, не знаю. Ну, вот только общемировые ситуации, да. Ну, как бы вот [СВО] – это плохо. Все. А в остальном нет, наверное. Как-то я просто в целом такой человек, что я могу там, не знаю, у мамы, у подружек совета попросить, все равно сделаю по-своему. И когда я сделала, даже если я подумаю: «Ну, наверное, да, я поступила неправильно», – но на тот момент мне нужно было поступить именно так. И особо никогда ни о чем не сожалею. Поэтому нет. Больше того, что зависело от меня, не, я, наверное, ничего бы не поменяла.
Интервьюер: Угу. Еще ты уже, ну, много говорила, да, про какие-то сложные, трудные ситуации в жизни. Вот. Но коли у меня есть такие вопросы, я должна их задать, да. Расскажи, пожалуйста, в жизни в целом, не только за последние два-три года, а в целом, какие периоды, какие ситуации ты можешь назвать трудными, тяжелыми? Вот что для тебя вообще какая-то трудная, тяжелая ситуация?
Информант: Ну, у меня детство воспринимается достаточно сложно, потому что у нас, именно когда я была совсем мелкая, как бы особо ничего не понимала, у нас были сложные взаимоотношения внутренние между взрослыми, между мамой, тетей, бабушкой. Вот. И, ну, это все на мне отражалось. И у меня было ощущение, что я какая-то ненужная, второсортная в семье. Вот. Оно потом, я до сих пор над этим с психологом работаю. Вот. Потом уже постепенно, когда там сестра у меня выросла, ушла, там вышла замуж, ушла, грубо говоря, у нас из семьи, то как-то больше все-таки семьей стала моя мама и моя бабушка. Вот. А остальные ушли на второй план. Мне стало спокойнее, уже не было такого ощущения. А вот совсем маленькая, да. В целом, наверное, пока я не выросла, очень сложно воспринималось, тоже я не могу назвать это каким-то периодом, воспринималось отсутствие отца. Вот. Я его никогда не знала, знаю только имя, но как бы даже не видела его никогда, ни разу в жизни. И у меня всегда вот было какое-то такое детское представление, что вот там на день рождения вдруг он появится, или там на выпускной вдруг он появится, или еще что-то такое. Вот. Ну, как бы он так и не появился. Лет в 20 уже стало все равно, потому что уже как бы даже если бы он появился в моей жизни, папы у меня все равно не будет. То есть я уже выросла без отца, и это не изменится. Вот. А именно событие – наверное, когда бабушки не стало. Это, ну, да, тяжелый удар был для всех. С одной стороны, все прекрасно понимали, что ей уже много лет, и она очень много инсультов перенесла, инфаркт, и как бы, ну, много проблем со здоровьем в целом было. Вот. Все прекрасно понимали, что рано или поздно это произойдет. Но это, знаешь,
Интервьюер: Это всегда неожиданно.
Информант: Да, да. Вот. Поэтому, да, было в этом плане очень тяжко. Вот. Вся семья переживала. Но, с другой стороны, вся семья объединилась. Вот. Был сложный период, когда, бабушка еще, по-моему, была жива на тот момент, я жила со своим братом старшим. То есть у меня там сложная ситуация получилась. Тете нужно было на всякий случай снять квартиру в Питере. Вот. Потому что у них была служебная с дядей, и их могли в любой момент оттуда выселить. Вот. И мы договорились, что в той квартире буду жить я, половину за квартиру оплачивать. Вот. Половину тетя, потому что полностью я не могла оплачивать эту квартиру. Вот. И в какой-то момент брат решил туда переехать, это вот сын моей тети. Вот. И в общем-то мы начали жить вместе, и у него было, ну, неадекватное поведение. И очень один раз меня задело, когда он, ну, поднял на меня руку, и мой парень его остановил. На тот момент парень. Вот. И, ну, как бы все, я собралась, оттуда уехала, сказала, что я не буду там жить. Но я очень тяжело воспринимала, потому что у меня было ощущение, что меня как бы вот брат предал, не знаю, что-то такое. Вот. Так был сложный период, тоже вот с тем же парнем мы жили, когда он попал в больницу, я осталась без работы, ему нужно было в больницу тогда таскать еду, а денег не было. И я там че-то две недели гречу одну ела, потому что, ну, все как бы, все остальное в квартире закончилось. Просить у мамы мне гордость не позволяла, всякое такое. Вот. А как бы заработать, ну, я не могла. У меня даже не было денег, чтобы доехать куда-то, чтобы пройти собеседование. Вот. Потом вроде как, я уже не помню, как я выкрутилась из этой ситуации, но как-то доехала до собеседования и куда-то там пристроилась как раз вот на первое время. Ну, тоже, да, такой, наверное, сложноватый период был. Сейчас уже плохо вспоминается. Я не помню, как я это ощущала, но просто помню, что, ну, как бы просто было тяжело. Вот. Как-то так. Больше вроде ничего не вспоминается.
Интервьюер: А вот, ну, когда в целом тебе плохо, да, или когда какие-то трудные ситуации, как вообще стараешься с ними справляться? Есть у тебя какие-то способы отвлечения, справления?
Информант: Ну, смотря какие сложные ситуации. Если это что-то вот а-ля из личной жизни, я могу себе дать немножечко времени там пострадать, поплакать, не знаю, послушать грустную музыку в мыслях о нем, там всякое такое [смех] Вот. Но в какой-то момент отряхнулась и пошла дальше. Вот. А если это вот как сейчас, когда вот мне было тяжело, мне вот, как оказалось, нужно было время просто наедине с собой, чтобы меня никто не трогал и ничего не заботило. Вот. Ну, то есть, да, наверное, побыть один на один с собой. Когда до этого были сложные моменты, очень часто я уходила с головой там в свои увлечения, в работу. То есть у меня были периоды, вот как раз когда бабушки не стало, я поступила в университет, я параллельно там работала, я параллельно там, не знаю, танцами занималась, еще там чем-то занималась. То есть у меня все расписание было просто минута в минуту, я там лишний раз не могла в туалет сходить, потому что не успела бы куда-нибудь. Вот. И это позволяло мне просто, ну, не думать, не переживать о том, что вот бабушки не стало. Вот. А так, видимо, вот зависит от ситуации. То есть по-разному ситуации переживаю. Но у меня всегда, всегда, что бы ни произошло, даже мелочь какая-то, мне сразу надо кому-то позвонить и рассказать. То есть либо там подружке, либо маме, но вот это выговориться – это обязательно, это всегда помогает. Вот.
Интервьюер: Поняла. А если говорить про какие-то хорошие, да, моменты, то что вообще в жизни тебе доставляет радость, удовольствие?
Информант: Собак очень люблю [смех]
Интервьюер: У тебя есть собака?
Информант: Нет. Я вот я уговариваю парня завести собаку. Он боится очень брать эту ответственность. Я понимаю его прекрасно. Но у меня прям все зудит во всем теле [смех] Настолько сильно хочется собаку. Вот. Мечта, наверное, всей жизни. Вот. В целом мне вот сейчас очень нравится, чем я занимаюсь. То, что я занимаюсь фотографией, прямо мне доставляет именно сам процесс удовольствие, взаимодействие с людьми. Мне нравится, когда я снимаю, запечатлевать именно эмоции какие-то у людей. Вот. Я стараюсь их как-то рассмешить, еще что-то, да, там какие-нибудь шуточки-прибауточки рассказать, чтобы они там начали друг с другом взаимодействовать больше, смеяться, и ловить вот эти моменты. Мне очень нравится, когда что-то такое получается, и кадр прямо такой живой выходит. Вот. Вот этот процесс мне очень нравится. Мне очень нравится ощущение, когда я там занимаюсь йогой, когда я занимаюсь спортом. Вот… Напомни вопрос, пожалуйста [смех]
Интервьюер: Про то, как, что тебе доставляет удовольствие, радость в жизни.
Информант: Наверное, вот еще ээ времяпрепровождение с сестрой, с мамой, с друзьями. Как-то очень, очень ценю свой круг общения. Вот. Мне кажется там люди прямо подобрались вот четко как вот под меня. Вот. Сейчас меня все устраивает в моем окружении на 100%, и очень люблю с ними время проводить. Вот. Вот когда день рождения последний раз было, отмечали, прямо мне очень радостно было, что вот все близкие мои друзья собрались, и все столько теплых слов сказали, у меня прям такая радость на душе была. Вот. А так в целом не знаю. Даже просто банально сериал полежать посмотреть. Или когда очень сильно устал, пришел домой, такой снимаешь лифчик, ложишься на кровать и такой: «Боже, жизнь прекрасна. Что может быть лучше?» [смех]
Интервьюер: Да, это точно [смех] Маленькие женские радости. А что уверенность тебе придает?
Информант: Ой, уверенность. Наверное, когда не приходится заботиться о том, сколько у меня там денег осталось до там, не знаю, ну, до этого, когда я работала, там до зарплаты или сейчас там до зарплаты парня, чтобы лишний раз там у него ничего дополнительно не просить. Вот. Финансово все равно как бы, сколько бы я ни говорила там и парню, и друзьям, что меня деньги не волнуют, деньги – это неважно, и я готова там одевать в секонд-хенде. Это так и есть, но при этом как бы когда денег на карте, там, я не знаю, 1 000 рублей остается, ну, чувствуешь себя немножко не в своей тарелке. То есть в этом плане, да, это важный аспект. Вот. Уверенность придает поддержка. Когда вот сейчас у меня прямо очень в плане той же фотографии у меня бывают какие-то такие моменты, когда я думаю, что, блин, я нигде не училась на это, я же так вот на Ютубе видео смотрела, там один курс прошла, и все. Типа что я об этом знаю, грубо говоря, да. Вот. Когда вот ездили в Казань, подругу фотографировала там на фоне дворца на телефон, она такая: «Боже, ты даже на телефон так классно фоткаешь!». Там: «[Друг молодого человека], посмотри, посмотри, как она круто фоткает!» Я думаю: «Боже мой, как приятно» [смех] То есть вот какие-то такие моменты. Какая-то обратная связь от там тех же заказчиков, когда им нравятся работы, тоже придает уверенность, что все-таки я не просто так беру за это денежку, все-таки оно того стоит, наверное, для людей. Вот. Ну, да, поддержка близких, тоже когда мама, друзья поддерживают, когда говорят какие-то приятные вещи, прям очень, очень так, ох, радостно становится на душе. Вот.
Интервьюер: А есть ли что-то, чего ты боишься?
Информант: Змей.
Интервьюер: Змей?
Информант: Да [смех] Это с детства просто фобия. Вот. Они у нас водились на даче. Я все лето на даче всегда проводила. Их там было достаточно много. Ну, там гадюки и ужи только. Вот. И у меня какая-то дикая фобия просто, змей я даже на картинках смотреть не могу. Вот. А так именно каких-то таких страхов а-ля вот не иррациональных, как те же змеи, а достаточно рациональных, не знаю, ничего в голову не приходит. Наверное. Ой, не знаю. Вот я в какой-то степени очень хочу детей, и у меня, наверное, есть страх испортить им чем-то жизнь, знаешь, там психологическую травму нанести, еще что-нибудь. За счет того, что у меня не было отца, для меня теперь очень важно, чтобы у моих детей был прямо стопроцентный, там супер классный папа. Вот. И вот с этим выбором я боюсь очень ошибиться, потому что, ну, как-то так им должно повезти, я должна для этого все сделать. Вот. Я уже парню сказала, что если так сложится, и у нас будут дети, и мы разведемся, я говорю: «Ты не отвертишься, ты будешь». Я не буду, как вот сейчас у его друга лучшего жена там она родила ребенка, они развелись. Вот. Она ему там запрещает видеться с дочкой, всякое такое. Я говорю: «Нет, я тебя буду силком притаскивать за шкирку. Вот ребенок – общайся. Ты должен быть для него лучшим папочкой, даже если ты там самый плохой человек на свете, все равно». Вот. Какое-то такое есть. А так, ну, страх банальный, который, наверное, есть абсолютно у всех, там за родственников, там чтобы, не дай Бог, кто-то там не умер, не заболел, еще что-нибудь такое. Вот. Ну, нету такого, что это прям, не знаю, какой-то такой навязчивый страх, да. Ну, просто как бы просто обычный страх. Появился как раз, когда у мамы инсульт случился. Она мне еще тогда звонила. Вот дотянулась до телефона, первая кого набрала – меня. Вот. А я тогда только пришла домой, снимала обувь, стояла в куртке, я такая вижу, мама звонит, думаю: «Так, сейчас, это, разденусь и перезвоню». Вот. Раздеваюсь и вижу, уже звонит тетя. Вот. Я думаю: «Так, если». А тетя мне просто так не звонит. Вот. Я вижу и такая: «Так, что-то точно не то». Тут я уже беру трубку, она говорит: «Вот, что ты матери не отвечаешь? Там ей плохо, срочно одевайся и выезжай». Вот. И у меня теперь все, я не могу не взять трубку, когда мама мне звонит. Это вот прям: «А вдруг что-то случилось?» Вот. То есть после этого такой страх есть. Ну, то есть как-то такие достаточно обычные, мне кажется, вещи, которые так или иначе у всех присутствуют. Вот. Как-то так.
Интервьюер: Поняла. Ну, все, я надеюсь, закроем тему с какими-то трудностями. Давай поговорим вообще про фотографию, про твое дело. Как ты в этом вообще развиваешься? Как ищешь заказчиков?
Информант: Очень медленно развиваюсь [смех] Ну, вот опять-таки из-за депрессии. У меня как-то это все так одновременно сложилось, что, с одной стороны, наконец-то начала заниматься тем, чем всегда хотела, а, с другой стороны, внутренние ресурсы, очень мало энергии. Вот. Поэтому заказов у меня мало. Я в целом не брала каких-то сторонних заказов а-ля, не знаю, там где-то на сайтах не выкладывалась, объявления не писала. То есть у меня все, кто сейчас приходит, это вот сарафанное радио чисто. Это либо там мои друзья, знакомые, либо знакомые знакомых, которые там как-то от меня про меня от кого-то услышали, и вот: «О, нам нужен фотограф. Вот типа давайте». Вот. Как-то так. В целом я пока что, ну, не ставлю какой-то большой супер там ценник, потому что понимаю, что все равно я начала этим заниматься только там в ноябре прошлого года, то есть, ну, как бы считай там полгода прошло. Вот. Считаю, что у меня еще, ну, недостаточно опыта. Портфолио не такое большое, чтобы я могла прям большой ценник ставить. Вот. Но в целом сейчас я уже вот постепенно такая, я вроде чувствую себя лучше. Вот. Я уже набрала достаточное портфолио, чтобы куда-то выкладывать объявления, сейчас планирую в общем-то этим заниматься. Вот. Не знаю, на Авито выложить опять-таки, там в группах ВКонтакте, всякое такое. Сделать группу ВКонтакте в целом, чтобы, ну, она у меня есть, но там как бы ничего нету, без наполнения. Вот. То есть чтобы было как какая-то визитная карточка, что-то такое. Вот. Потому что пока кроме Инстаграма особо ничего нет. Вот. Но при этом как бы люди спрашивают. Даже вот я последнее то, что снимала, семейную съемку проводила, девочка, которая там была, у них с молодым человеком скоро свадьба, она такая: «Вот, нам на свадьбу нужен фотограф. Может быть, вот тебя?» Я говорю: «Слушай, у меня нет опыта в этом. Я боюсь что-то обещать». Она говорит: «Ну, покажи свои работы. Если мне понравится. В целом типа у нас небольшая свадьба, нам там не нужен именно прям свадебный фотограф и вот прям на все мероприятие». Она такая типа: «Вот скинь там свою страничку ВКонтакте». А я не знаю, у меня ничего нету, только в Инстаграме. Думаю: «Блин, надо сделать. Надо сделать». Вот. В плане того, как я этим занялась. Мне как-то, я уже говорила, еще в школе интересно было, очень хотелось. Когда я заинтересовалась астрономией, я думала: «Я буду астрофотографом. У меня будет куча оборудования, я буду выезжать по ночам там куда-нибудь на холмы и снимать звезды, ночное небо». Вот. Для этого нужно очень много денег, оборудование стоит супер дорого. Поэтому пока что все, что есть, это фотоаппарат. Вот. Ну, а дальше там посмотрим. Не знаю. Вот. Мне в целом всегда нравилось снимать. Я так или иначе интересовалось этой идеей. Но мне было интересно снимать именно людей. То есть вот у меня нет интереса снимать, как вот у нас общая подруга любит там снимать мимо проходящие какие-то детали, она прямо круто их подмечает. Я в восторге, честно говоря, от этого, как она ведет свою страницу в Инстаграме, мне очень нравится. Я так не умею. Мне нравится: вот я взяла фотоаппарат, передо мной сидят люди, я их развлекаю, тормошу и пытаюсь запечатлеть эмоции. Вот мне такой формат нравится. Либо там, я не знаю, а-ля вот происходит между ними какой-то диалог, и я подлавливаю, что-то снимаю. Вот. А какие-то там а-ля интерьерные вещи, там природу, что-то такое мне не очень интересно. Вот. Да. Не знаю, что еще сказать.
Интервьюер: Угу. Ну, так скажем, что ты начала, наверное, не в самое простое время. Ноябрь прошлого года. Вот. Как ты думаешь вообще вот ситуация там с ковидом и с СВО, как они вообще на этой сфере отразились? Появились ли какие-то дополнительные ограничения или, наоборот, возможности?
Информант: Угу. Слушай, ну, с одной стороны, вот я смотрю очень много фотографов, на которых я подписана, из России изначально очень много уехало. Мне кажется это вот фотографы, айтишники, там видеографы и так далее – это вот основная ээ масса населения, которые поуезжали. Вот. Поэтому, с одной стороны, рынок освободился немножечко [смех] и как бы, ну, фотографов, действительно, стало меньше. Хорошего фотографа, за тем более небольшую стоимость, найти всегда сложно. Вот. То есть поэтому на этом можно, действительно, как-то сыграть, грубо говоря, да, и найти себя достаточное количество заказчиков. Вот. В остальном мне кажется спрос не понизился. То есть у меня нет такого ощущения, потому что даже по своим знакомым у меня наоборот ощущение, может быть, просто потому, что я больше в этой теме кручусь, но ощущение, что спрос только повысился. Я не знаю, может, людям как-то сейчас стало важнее запечатлеть какие-то моменты, да, в связи там с ситуацией и так далее. Вот. Не знаю, это чисто вот такие предположения. Вот. Но я вижу, что спрос действительно большой. Даже вот как бы я не пытаюсь как-то развиваться пока что, да, то есть практически там никуда не выкладываюсь, ничего, а от знакомых мне приходят, ну, два заказа в месяц мне приходят сейчас. При том, что я не прилагаю никаких усилий. Вот. Поэтому не знаю сколько, мне кажется ребята, которые уже работают плотно в сфере и как бы действительно прям они профессионалы своего дела, они мне кажется просто вот так вот отбиваются от заказов, потому что больше не успевают. Вот. Надеюсь, у меня также будет [смех]
Интервьюер: Я тоже надеюсь, что у тебя все получится. А вот если говорить про именно какое-то оформление, ну, знаешь, там типа самозанятости и все такое. Ты вообще думала об этом? Для тебя это важно?
Информант: Да, конечно. Важно. Ну, я как бы хочу быть сознательным гражданином, да. Я понимаю, что налоги платятся не просто так, и как бы их надо платить. Просто сейчас у меня не такой доход, чтобы оформлять что-то. Вот. Но я планирую, как только у меня будет именно какой-то, как сказать, когда я пойму хотя бы какой-то постоянный свой доход, даже если это будет, не знаю, 15, 20 тысяч в месяц, я оформлюсь как самозанятая. Да, я буду, конечно, оплачивать налоги, все это понятное дело. Вот. Просто пока что у меня там был один месяц, когда я вообще ничего не заработала, был один месяц, когда я там не помню сколько, но у меня там было че-то три или четыре заказа, что-то такое, и что-то я хорошо там заработала для себя на этот момент. Вот. И, ну, как бы вот эта вот разница, когда у меня то ничего, то что-то. Пока непонятно просто как с этим подходить к оформлению самозанятости, налогов. Когда все более-менее прозрачно станет, тогда уже, да, обязательно. Вот.
Интервьюер: А если говорить про вот эту сферу именно самозанятых, да, то слышала ли ты о каких мм как ты в целом оцениваешь как бы развитие этой сферы в России, да, и слышала ли ты о каких-то мерах господдержки? Считаешь ли ты их эффективными?
Информант: Честно, про господдержку, наверное, просто даже не интересовалась. Вот. Наверное, именно когда я возьмусь за этот вопрос, конечно, я все изучу и почитаю, и как бы и буду разбираться и шарить в этом. Вот. В остальном я просто знаю, что самозанятых достаточно много. У нас в принципе сейчас люди, как мне кажется, как бы, знаешь, это такая идеальная картина мира, когда человек из офиса переходит там, знаешь, и не работает больше на дядю, а-ля вот что-то такое. И как бы по крайней мере пропагандируется там в тех же соцсетях, что вот все так хотят, давайте все к этому стремиться. Вот. Поэтому мне кажется, что так или иначе, действительно, многие переходят в самозанятость: кто-то там тортики печет, там, не знаю, макраме вышивает и так далее, кто-то вот фотографией занимается. Вот. Не вижу в этом, ну, ничего плохого. Мне кажется, все равно люди, которые работают на кого-то, тоже останутся, потому что далеко не каждый может работать сам на себя. Вот. Я до сих пор не уверена, что я могу. У меня, мне кажется, дисциплины, ну, не сильно хватает. Вот. Но я хочу попробовать работать над собой. В крайнем случае там тем же фотографом я могу устроиться и куда-то. Вот. Такие мысли тоже есть. Поэтому посмотрим. Я уже даже смотрела вакансии фотографа в разных компаниях. Вот. Тоже как вариант. Если я пойму, что все-таки я сама на себя работать не могу, не стану оформлять самозанятость. Вот. Как-то так. Вроде ответила.
Интервьюер: Угу. Да, все хорошо. А какие у тебя в целом вот именно планы на какое-то свое профессиональное развитие? Может, есть какие-то такие поинты, которых хотелось бы добиться?
Информант: У меня сейчас пока вот, как сказать, перед депрессией у меня прям, знаешь, четко все было распланировано. То есть я понимала там, что я делаю сейчас, что я делаю потом, к чему я стремлюсь, как к этому прийти, что мне нужно сделать. И вот просто все там по месяцам было расписано. Началась депрессия, мне пришлось взять академ, и собственно все, весь план рухнул, потому что там все, знаешь, одна цель цеплялась за предыдущую и так далее. Вот. По цепочке. Поэтому сейчас у меня просто одна вот эта вот отложенная немножко на данный момент цель – закончить универ, и развиваться в фотографии так, как я это представляю. Что из этого дальше получится, чем я дальше в итоге буду заниматься, я уже просто не хочу загадывать. Потому что когда очень на очень дальний срок загадываю, и потом в итоге это не складывается, очень сильное разочарование наступает и ударяет больно. Вот. Поэтому пока больше так загадывать не хочу. Тем более время у нас щас нестабильное вообще [смех] Это из рода шуток там, что у нас коридор планирования – там день [смех]
Интервьюер: Да, да, да. Нет, я как бы тоже дальше двух дней, я вообще.
Информант: Да, да, да.
Интервьюер: И там люди, которые планируют отпуск, покупают билеты заранее. Я вот сейчас вообще думаю: «Ну, вы чего?». В Москву мы собирались, мы такие типа, мы до последнего были не уверены, что мы до Москвы даже доедем.
Информант: Ну, вот мы в Казань собрались, как это, у подруги соревнования, ее парень нам такой типа: «Че, поехали с нами? Поддержим, все дела». Вот. Это было за день до поездки. Вот. [Молодой человек] мне там днем такой пишет: «Ну, что ты хочешь поехать в Казань?» Я говорю: «Да, хочу» – «Ну, [друг молодого человека] тогда возьмет нам сейчас билеты». Вечером [друг молодого человека] берет билеты, утром мы вылетаем, все. Ну, то есть как бы так вот, потому что иначе мне кажется невозможно, действительно, что-то планировать. Вот. Поэтому пока что просто просто делаю, что делаю. А дальше будь, что будет. И все. Вот.
Интервьюер: Поняла. У меня, наверное, еще, не знаю, минут 20. Все нормально, ты не устала?
Информант: Да. Нет.
Интервьюер: Вот. Сейчас тоже немножко сменим тему. Поговорим в принципе про поколения. Людей, наверное, какого возраста примерно ты вот считаешь своим поколением?
Информант: Ну, слушай, наверное, разброс большой, потому что я встречала людей прям супер разного возраста и с разными возрастами ладила, как-то находила общий язык. Я, наверное, не знаю, если так по общепринятым оценкам, то, наверное, вот там а-ля 95-ый тире там 2003-ий год. То есть я сама 98-го, ну, вот там плюс-минус там, сколько, пять пять лет. Не знаю, типа того. Вот. Такой разброс. А по ощущениям я спокойно могу найти общий язык и какие-то общие темы там в детстве, да, и в становлении и с людьми там возраста моей сестры, которая родилась в 86-ом. Вот. И при этом там у меня были знакомые, которым там вот-вот 18 исполнилось, мы тоже вполне находили общий язык. То есть, ну, люди разные. Вот. Наверное, очень сказывалось ээ наличие там тех же компьютерных технологий, да, в жизни. У меня они появились достаточно поздно. Поэтому как бы я я больше похожа на более старшее поколение [смех] чем на свое. Вот. В этом плане. Так что как-то так. Не знаю. Но большая частью друзей при этом у меня плюс-минус моего возраста, то есть это вот 25, ну, тире там 30 лет. Вот так вот.
Интервьюер: Угу. А есть какие-то особенности или характеристики вот именно этого поколения? Какие мы?
Информант: Мне кажется да. Ну, как-то большинство особо не стремятся там вот срочно, прямо сейчас там жениться, замуж, там детей и так далее. Вот. То есть как-то больше сосредоточены даже не на карьере, а на себе, на том, чтобы сложить все для себя вот так, как хочется, да. То есть не просто чтобы там выйти замуж, а чтобы выйти замуж вот по любви, чтобы все было идеально. Вот на какой-то такой идеальности что ли. Там если кто-то хочет путешествовать, то как бы работа, не работа, карьера, не карьера – пофигу, мы будем все делать, чтобы путешествовать. То есть какое-то такое, наверное, больше сконцентрированность на себе. Это может, наверное, даже звучать как-то, ну, не очень, как-то эгоистично, а мне кажется, что это прекрасно. Почему бы нет? Вот. Жизнь у нас одна собственно. Вот. Как-то так. А в остальном при этом мне кажется, что поколение у нас достаточно трудолюбивое при этом как бы. То есть если человек чего-то хочет добиться в карьерной сфере, ну, человек действительно может приложить там все усилия и добиться этого. Вот. Потому что возможностей супер много сейчас. Даже там при нестабильности нынешнего времени все равно возможности там и переехать есть, хоть и сложнее стало, и там что-то изучить гораздо проще, чем те же 10 лет назад. Вот. Поэтому, да, возможностей очень много, и как бы и люди их хватают, и делают. Вот. Не знаю, наверное, все.
Интервьюер: Угу. А если вот сравнивать, да, например, твое поколение с поколением более старшим, ну, например, там твоих родителей, бабушек, дедушек, вот есть ли какие-то отличия?
Информант: Мм ну, вот если сравнивать конкретно меня с моей мамой, разницы практически нет. Вот. Мы, ну, очень похожие люди. Она тогда, наверное, не шибко вписывалась в общество того времени. Вот. То есть как бы она не торопилась выйти замуж, родить детей, она жила в свое удовольствие. Вот. И потом к 35 уже такая: «Ну, можно бы и дитятку родить». Вот. А в остальном, если сравнивать с другими людьми, да, там вот ее сестра в 18 замуж вышла, сразу ребенка, то есть как-то больше все-таки, наверное, был упор на, не знаю, на демографию. Вот. То есть у людей, как мне кажется, опять-таки чисто мое мнение, выходцев из СССР, да, которые все-таки становление проходило в советское время, у них было четкое представление, что дальше, да, была вот эта какая-то стабильность, а-ля государство нам даст, не знаю, комнату в коммуналке или квартиру. Вот. Нам дадут там место на заводе и так далее, да. То есть ты пошел отучился где-то, да. С каждого по способностям, каждому по потребностям, да. И вот все люди примерно так спокойно жили, особо никуда там не смотрели, не рвались. Вот. У нашего поколения все-таки все по-другому. Нам не дадут квартиру просто так. Нам не обязательно так просто получить место там где-то на работе и так далее, да. Но при этом у нас гораздо больше возможностей в плане, как сказать, чему-то научиться новому, куда-то поехать, что-то посмотреть, составить свое мнение, там высказать свое мнение даже. Вот. Как-то так. На менталитет это тоже очень влияет, мне кажется именно в плане, как сказать, раньше людям запрещали верить в Бога, да, они там тихонечко, в тайне сидели молились, сейчас людям не запрещают верить в Бога, а они как бы не будут этого делать, грубо говоря, так. Вот. Как-то так.
Интервьюер: Угу. Я поняла. А если вот подумать про тех, кто помоложе, то тоже, может быть, можешь что-то назвать?
Информант: Ну, вот у меня есть племяшка, ей 9. Не знаю, наверное, могу на ее примере пре представить. Ээ музыку слушают странную [смех] Уже начинает появляться вот эта вот фигня а-ля «а в наше время». Вот. Но при этом что меня поражает, у детей нынешних у них как будто меньше табу что ли. То есть мелкая там те же, там ей шесть лет когда было, она спокойно со мной могла поговорить на какие-то откровенные темы, которые мне в ее возрасте, я тоже задумывалась об этом, у меня тоже было какое-то мнение, но я боялась об этом говорить со взрослыми, потому что это было табуировано, даже если напрямую не говорили, что об этом нельзя говорить, это чувствовалось. Там те же разговоры про религию. В моей семье там сказать, что я сомневаюсь в том, что есть Бог – это было, ой, ты что, это все, бабушка сразу тебя святой водой вот так со шланга обольет. Вот. А у меня с мелкой спокойно, у нас были доверительные разговоры на эту тему. Там и сейчас в девять лет она уже знает, что такое месячные, она как бы готова к этому уже морально. Потому что, а почему нет собственно? Вот. Как бы когда я была в ее возрасте, все боялись разговаривать на эти темы. Как-то так. Поэтому у них мне кажется растет какое-то более свободное поколение что ли. Вот. Дай Бог, я на это очень надеюсь. Вот. Мне не нравится там нынешняя вот, как сказать, антипропаганда ЛГБТ-отношений, да, у нас в стране, потому что я не вижу в этом ничего предосудительного. Как бы люди также такими же рождаются, да, там те же трансперсоны. Как бы, ну, человек не виноват, что он такой, почему это нужно запрещать на законодательном уровне, я не очень понимаю. Вот. Поэтому я надеюсь, что как раз-таки вот это вот более свободное поколение, которое сейчас подрастает, они изменят вот эту парадигму восприятия у нас в обществе. Вот. Может, появятся однополые браки. Я была бы очень рада за других людей, у которых бы получилось пожениться, не пришлось бы для этого уезжать в другую страну, где это можно. Вот. Как-то так.
Интервьюер: Я поняла. А если возвращаться опять-таки к ситуации, да, прошлого года, то как ты думаешь, как она отразилась именно вот на твоем поколении?
Информант: На моем что?
Интервьюер: На твоем поколении.
Информант: Именно ты имеешь в виду ситуацию с СВО?
Интервьюер: Угу.
Информант: Ну, очень многие уехали. Однозначно. Причем я часто слышу от знакомых, что там говорят, что типа: «Ой, у меня все, кто уехал, все вернулись». У меня вернулась только сестра, и не по своему желанию, потому что пришлось вернуться. Остальные никто не возвращался и как бы и не хотят этого делать. Вот. Есть немножечко ощущение, что общество как бы разделилось на тех, кто за, и тех, кто против. Вот. То есть есть есть люди, конечно, которые такой пытаются сохранять нейтралитет, но как бы, ну, это сложно, потому что все всех волнует твое мнение по этому поводу, чтобы отнести тебя к той или к иной категории. Вот. Чтобы как бы, грубо говоря, приписать к своим или к чужим. Вот. Это тоже не есть хорошо вот это вот такое разделение в обществе. Вот. Многим людям, в том числе и мне, которые до этого были такие аполитичные, типа меня политика не волнует, я в это дело лезть не буду, пришлось так или иначе хоть какое-то мнение о политике составить. Потому что, ну, просто а как иначе? Тут именно времена заставляют это делать, как-то поинтересоваться, той же историей начать интересоваться и так далее. Вот. Аа наверное, это одна вот из немногих хороших вещей, что людям пришлось задуматься о политике. Вот. В остальном, конечно, позитива маловато. Очень маловато. Вот. Не знаю, наверное, все, что вот так вот в голову приходит.
Интервьюер: А если, ну, говорить опять же про поколения, как ты думаешь, кому сейчас проще, кому сложнее? Тем, кто постарше, ну, условно вот нам или тем, кто помладше?
Информант: Мм не знаю, мне кажется всем сложно и просто по-разному, да, в зависимости от возраста. Тут дело мне кажется даже не в поколении, а именно вот в силу возраста, да, люди так или иначе успевают там, я не знаю, поколение моей мамы, ей сейчас 60, да, 50 даже кому, у них уже четко выстроено мировосприятие. И чем старше человек, тем, насколько я представляю, да, знаю, ему сложнее поменять свое вот это вот отношение к происходящим ситуациям. Вот. В этом плане, наверное, им проще. То есть вот если там, я не знаю, поколение, там представитель поколения моей мамы, не знаю, привык восхищаться Путиным, он не перестанет им восхищаться. Ему за счет этого проще, потому что у него не возникает каких-то, как сказать, моральных каких-то, знаешь, вопросов в голове, что типа плохо это или хорошо. То есть ему не приходится колебаться, он знает, что Путин – это хорошо, Путин делает хорошо. Окей. Вот. А наше поколение, которые опять-таки многие были аполитичные, особо не задавались там молодец Путин или не молодец Путин, ему приходится составлять мнение, да, и как-то все-таки разбираться в этой теме, да. Для кого-то, может быть, там могут быть сложными именно, да, вопросы морали. То есть вроде бы как они поддерживают нынешнюю власть и государство, но при этом головой понимают, что [СВО] – это плохо. И типа куда себя отнести, не понимают. То есть какие-то такие вещи тоже среди своих знакомых встречала. Вот. В этом плане тяжело. В остальном людям старшего поколения очень тяжело, мне кажется, в плане того, что им бедным пришлось пережить развал СССР, 90-е, нулевые тоже были непростыми. Вот. Там кризис 2007-го или какого? 2008-го года. Вот. И вроде только-только спокойно стало, да более-менее, и щас опять какая-то фигня происходит. То есть в этом плане им тоже тяжело. Мы маленькими были, да, мы это особо не застали, не заметили, ну, росли себе и росли, как бы не приходилось выживать именно, да, в тех условиях. Там у меня маме в 90-е пришлось прямо супер туго, и в нулевые тоже. Вот. Я маленькая была, я такая, ну, че, поесть есть, там голая не хожу, грубо говоря, без обуви не хожу, ну, и все нормально. Вот. Как бы для нас это, с одной стороны, первый раз с этим столкнулись со всем, да, с такими сложностями во времени тоже очень тяжело, потому что ты немножечко не понимаешь, а как реагировать, а что делать, да, а как правильно. Вот. А старшее поколение уже такое на опыте, можно сказать. Но, с другой стороны, вроде только расслабились, и вот опять. Вот. Всем сложно и просто по-своему мне кажется.
Интервьюер: Поняла. А есть ли в этой ситуации что-то, чего твое поколение лишилось и что оно приобрело?
Информант: Ну, лишилась однозначно свободных поездок за границу. Опять-таки как я тогда поехала за границу. Я на машине, на Блаблакаре, доехала до Лаппеэнранты. Оттуда я полетела до Милана на самолете, до этого, до Бергамо. Самолет че-то стоил, перелет а-ля две тысячи, что-то такое. Оттуда на Фликсбасе я доехала, ну, сначала доехала до Милана, из Милана на Фликсбасе доехала до Женевы. Вот я, пожалуйста, туда-обратно путешествие из Питера до Швейцарии у меня обошлось в 9 тысяч рублей. Вот. Со всеми переездами. Там внутри тоже по Франции покаталась, по, Господи, ну, вот в Бергамо, в Милане погуляла еще на обратном пути. То есть как бы вот именно вот этого, да, лишились однозначно, потому что сейчас поехать туда же как бы куда-либо, да, мы не можем поехать в ту же Финляндию, там Ригу, ну, Латвию, Литву я имею в виду. Вот. Потому что эти страны для нас закрыты. Чехия, насколько я помню, тоже сейчас для туризма закрыта. Вот. Интересные страны, я бы очень хотела туда съездить, там побывать, тем более там до той же Нарвы, да, до Латвии, ой, вру, до Эстонии, это Эстония, можно было вообще пешком перейти просто через речку, и все, а оттуда, пожалуйста, в Таллин и гуляй себе на здоровье, прекраснейший город. Сейчас нет, это сложно, и это печально. В какой-то степени, так или иначе, как у нас есть люди, которые говорят, что украинцы – нацисты, их всех убить, порвать там надо, точно так же за границей есть люди, которые говорят, что русские – вообще не люди, их там всех выжечь надо. И это тоже, ну, влияет. Как бы даже если ты куда-то выберешься, на тебя там могут быть косые взгляды, и так далее. И просто, ну, может, и нет, может, ты на это не попадешь, может, все будут милыми и радушными, будут сочувствовать, но как бы не угадаешь. Из-за этого есть банально даже страх куда-то поехать. То есть я бы, например, очень хотела съездить в Испанию. Безумно люблю испанский язык, эту культуру. Вот. Там с парнем договаривались, что будем копить на эту поездку, хотя это будет очень дорого сейчас стоить, но все же. Вот. Но, ну, все равно есть страх, а как бы а как, а пропустят, не пропустят, а когда мы вернемся обратно. То есть у меня сестра, когда возвращалась с Кипра, им там полуторачасовой допрос устроили – почему они отсутствовали столько времени в стране, что они там делали, а почему. То есть пытались, ну, как бы их подловить на предмет попытки типа как уехать, знаешь, там что-нибудь против страны сказать и так далее. Вот. В этом плане тоже там страшно, как сказать, со стороны нашего государства тоже страшно, что оно может сделать, что оно еще может выкинуть. Вот. В какой-то степени свободы слова лишились. То есть как бы я не против нашей власти, но я против [СВО]. Но я не могу пойти открыто об этом сказать, потому что меня сразу в кутузку посадят. Это тоже страшно. Даже если не посадят в кутузку, банально я студент вуза, меня могут отчислить. Поотчисляли, ну, достаточно много народу. Слышала об этом. Как бы у меня знакомых никого не отчисляли, ну, именно вот с кем я училась, учусь. Вот. Но так вот по новостям слышала, там в СПбГУ поотчисляли, там в Меде, по-моему, отчисляли, там еще где-то, в этом, в Горном что ли, то есть, ну, в таких учреждениях. Вот. Все равно страшно. Боишься, ну, грубо говоря, за свою же жопу. Типа что с тобой дальше будет, если ты пойдешь и выступишь. Вот. Как бы сильно этого не хотелось – пойти и выступить. То есть вот этого лишилась, да. Я лишилась поездки в Украину. Я бы очень хотела туда съездить. У меня, ну, с Харьковской области вся родня по сути. Ну, с отцовской стороны я не знаю, кто откуда, я никого не знаю, а с материнской все оттуда. Вот. И в Харьковской области я никогда не была, хотя у нас там дом стоит до сих пор, в который заезжали мои бабушки. Вот. Я очень хотела туда съездить, посмотреть вообще, что там, как, может, как-то восстановить, погулять по местам, где моя мама росла в той или иной степени. Вот. А я не могу этого сделать. Там Одессу очень люблю, я не могу туда поехать, потому что там [СВО], потому что я туда не могу попасть. Вот. Там встретиться с родственниками теми же с Украины тоже не могу. То есть вот какие-то такие вещи. У нас же народы очень сильно друг с другом повязаны. У большинства людей, так или иначе, есть родня или друзья оттуда. Вот. Многие не могут сейчас друг с другом встретиться. Многие не могут нормально общаться друг с другом, потому что настраиваются пропагандой друг против друга. Вот. У меня там мама тоже со своей там двоюродная, по-моему, это ее сестра, она живет в Виннице. Вот. Это западная Украина. Они вроде бы стараются нормально контактировать и поддерживать общий язык, но при этом каждый раз, когда там тетя скидывает маме там какие-нибудь картинки а-ля там «Украина, вперед! Мы победим!», маму триггерит, она такая типа: «Ну, зачем ты мне это скидываешь?» Вот. Потому что мама у меня такая прямо за Путина, она такая: «Вот у него были причины, значит, это сделать, раз он это сделал». Вот. И как бы, она, с одной стороны, хочет с сестрой поддерживать хорошее общение, а, с другой стороны, вот у них вот такое вот перекидывание мемами превращается немножечко в такую мини [СВО]. Вот. Как-то вот так вот оно. Много как повлияло в общем-то, да.
Интервьюер: Угу. А как ты думаешь, что вообще будет с нами, со страной там в ближайшие пять-семь лет?
Информант: Я очень надеюсь, что [СВО] закончится. На самом деле… не знаю, может, так плохо говорить, надеюсь, не в нашу пользу. Потому что очень жалко Украину. Все-таки у меня все равно ощущение есть, что, ну, как бы им будет совсем плохо, если выиграет Россия, и я бы этого не хотела. Вот. С другой стороны, как бы тоже совсем, знаешь, жить в побежденной стране тоже не очень хочется, потому что очень, очень много чего плохого это за собой повлечет для нас. Вот. Не знаю. Я как бы, знаешь, надеемся на лучшее, готовимся к худшему. Вот и все. Потому что, знаешь, до февраля прошлого года я представить не могла, что у нас страна [СВО] разведет с Украиной. Вот. И как бы если бы мне кто-то сказал бы об этом, я бы у виска покрутила и сказала, что человек сумасшедший, и все, и пошла бы дальше. А потом хоп, это происходит, и ты такой типа: «Ну, такого же не могло быть». А, нет, могло. Вот оно случилось. Вот. Поэтому сейчас как-то мне кажется вообще невозможно представить, что будет через пять лет. Вот. Единственное, что я знаю, что ээ когда там, в марте или в апреле, в общем весной следующего года у нас будут президентские выборы. И вот это очень интересно, что там будет. Потому что, ну, как бы можно по-разному относиться к Путину, но факт того, что, ну, засиделся и что ему реально много лет – это, ну, факт неоспоримый. Да, можно считать, что он правильно поступает, но как бы мужику уже сколько там, за 70, по-моему, да, или около того, ну, многовато лет, пора бы все-таки ему смениться. И очень интересно вот, что будет происходить в этот момент. То есть, может, кого-то поставят, да. Ну, как бы он сам преемника своего назначит. Интересно, кто это будет и как это будет. Может, он все-таки переизберется, и он снова будет. Тогда, скорее всего, особо ничего не поменяется, потому что, ну, как бы много лет он уже у власти, и как бы мы видим, что и как у нас, понятно, чего ожидать, грубо говоря. Вот. А если он сменится, тогда вообще непонятно, чего ожидать, кто придет на его место, как оно дальше будет, то есть в этом плане, да, такой эффект неожиданности присутствует. Вот. Опять-таки я все равно не шибко разбираюсь во власти, да, и в политике, чтобы там, не знаю, думать о каких-то прогнозах, я, наверное, не взяла бы на себя такую ответственность. Вот.
Интервьюер: Все, мне кажется, точно эту тему уже закрыли.
Информант: Да, да
Интервьюер: Она почему-то постоянно появляется. Вот. У меня еще несколько
Информант: Наболевшее.
Интервьюер: Да. Как бы все равно контекст он очень сильно влияет.
Информант: Угу.
Интервьюер: Вот. Еще несколько вопросов в принципе про инициативность, активность. Считаешь ли ты себя активным человеком? И кто вообще для тебя тебя активный, инициативный человек?
Информант: Наверное, на данный момент не считаю. Вот. Потому что максимум чем. Ну, для меня, наверное, начну с того, что активный, инициативный человек, этот человек у меня сразу представляется в голове волонтер. Вот. Там или человек, который организовывает какие-нибудь там, тавтология, организации. Вот. Ээ… наверное, нет, себя я таковым не считаю, потому что единственное, что я делаю, это там периодически пытаюсь куда-то вот в какие-то, как сказать, какие-то проекты там вкинуть денежку, там хотя бы чуть-чуть, там бабушке на улице подать, там всякое такое. Наверное, на этом моя активность заканчивается. Единственное, что сейчас я вот пытаюсь найти какой-нибудь волонтерский проект для фотографов, то есть чтобы где-то по России поехать. Пока нашла только один, он мне не очень подходит по разным причинам. Вот. Поэтому ищу дальше. Но вот хотела бы как волонтер поехать и как-то, может, пригодиться в своей сфере. Вот. Тогда, наверное, да, я после этого смогу сказать, что я более-менее активный человек. Вот. А в остальном, к сожалению, пока нет. У меня есть желание там помогать приютам. Я много читаю про разные инициативы, но пока что вот, к сожалению, только читаю. Например, знаю одно заведение, щас, «Вход с улицы» называется. Слышала?
Интервьюер: Ночлежка, да. От ночлежки.
Информант: Вот. От Ночлежки. Я туда ходила, очень мне понравилось. Мы оставили там ээ материальную помощь для Ночлежки. Вот. Я пообщалась с сотрудниками, они рассказали про своих подопечных, про уже выпустившихся, про ныне работающих. Вот. Очень интересно. Мне бы хотелось как-то, наверное, участвовать в таких организациях, помогать так или иначе, не знаю, так же как волонтер, может быть. Вот. Но пока что, ну, понимаю, что просто, ну, просто не до того на данный момент именно в плане там той же депрессии. То есть я не всегда могут себя заставить, как я говорила, из дома там выйти лишний раз. Вот. Поэтому пока что понимаю, что не вариант. То есть что могу, насколько могу, делаю. Больше пока нет возможности и нет возможности. Но меня очень вдохновляют люди, которые этим занимаются, потому что, ну, мне кажется это прям вот действительно благое дело. То есть это ты там все равно, да, в какой-то степени волонтерство – это для себя тоже. Вот. Потому что, ну, как бы ты себя потом чувствуешь хорошим человеком, да, что ты где-то кому-то что-то помог, сделал безвозмездно. Вот. Но это в любом случае хорошее дело, оно все равно идет на благо кому-то. И вот это очень вдохновляет. Когда-нибудь тоже хочу так делать.
Интервьюер: А есть вот среди твоих друзей, знакомых такие активные, инициативные люди?
Информант: Ну, наверное, да. Но тут скорее не про волонтерство, да, как я это представляю. Аа в группе у меня есть ребята, один парень, который супер активный, инициативный вообще по всем фронтам мне кажется. Он и староста группы, и председателем профсоюза был, занимается танцами, там, не знаю, Господи, сейчас работает в школе учителем, в лицее, физику преподает. При этом достаточно хорошо учится на самом деле, то есть все прекрасно. Такой весь очень, знаешь, как сказать, очень коммуникабельный парень. У него везде налажены контакты, он вот все везде везде, куда ни глянешь, он везде присутствует короче. Вот. Не представляю, откуда у него столько энергии [смех] Мне кажется, я именно знаешь, как это, по уровню энергичности не такой человек, и настолько активной я бы просто, ну, физиологически не смогла бы стать. Вот. Вот, наверное, что-то такое. Потом еще, Господи, ну, с [первый институт], да, у меня были знакомые ребята, которые опять-таки в [первый институт] там во всяких активностях присутствовали, там студенческих и внестуденческих, в том числе от вуза волонтерские организации. Вот. Единственное, я там вот именно, ну, не совсем волонтерство, что я там делала, я была в проекте по сдаче крови на донорство. Вот. Сейчас хочу к этому вернуться, но чуть-чуть попозже, потому что надо сначала со своим здоровьем все-таки разобраться. Вот. Аа но те, с кем я тогда училась в [первый институт], вот кого я знала, они сейчас поуезжали опять-таки. Вот. Очень мало кто остался в России, тем более там вот именно в Питере. Вот. Наверное, все. В остальном больше никто особо на ум не приходит. Вот. Аа вру, ну, вот опять-таки тот же [друг молодого чеовека], про которого я уже упоминала. У него достаточно много денег. Он особо, я так понимаю, не афиширует, но так или иначе они стараются помогать. То есть там и беженцам с Донбасса они стараются финансово помогать. Тоже, ну, есть одна ситуация, наверное, не буду так подробно рассказывать, потому что все-таки, ну, не моя история. Вот. Но очень хорошо он помог одной девушке как бы тяжелобольной. Вот. Там даже я не знаю про финансовую помощь, я не про финансовую сейчас вообще, именно моральную поддержку, которую он ей оказал, просто колоссальную. Вот. То есть вот какие-то такие вещи он делает. Это вот, насколько я знаю, достаточно тихо, то есть нигде не распространяется. Просто потому, что может. Особо этим не гордится, просто делает, и все, потому что считает, что так надо и так правильно. Вот. В этом плане я им очень восхищаюсь. То есть, наверное, да. Вот так вот.
Интервьюер: А если говорить про именно вот какую-то предпринимательскую, да, какую-то бизнес-активность, то какие вообще у тебя мысли на счет именно открывания своего дела сейчас в России? Самой или когда другие открывают.
Информант: Ээ как сказать [смех] Достаточно, с одной стороны, достаточно просто. Но смотря, смотря в какой отрасли. То есть, например, стать самозанятым, да, вот мне сейчас как фотографу, проще простого, особенных сложностей я не вижу. Там на госуслугах можно банально, да, это все оформить, и все. Вот. А открывать какой-то серьезный бизнес, ну, это всегда риски, это всегда сложности. Вот. Но при этом я вижу, как это у людей получается. Опять-таки этот человек должен быть определенного склада ума. Не каждый к этому готов, не каждый с этим справится, мне кажется. Вот. Но, тем не менее, люди делают, у людей выходит так или иначе, и это прекрасно. Потому что мне кажется у нас сейчас очень много, ну, малого бизнеса, да. То есть те же, как сказать, кафешки, да, рестораны в период ковида пострадали очень. Вот. Но при этом выходишь на улицу, очень много новых заведений открывают и действительно крутых. Мне кажется у нас эта сфера очень здорово развивается сейчас. Вот. В плане там тех же едальных, питейных заведений очень, очень круто все преображается. Вот. Это же тоже кто-то делает, кто-то открывает, да, вкладывает в это средства, прорабатывает бизнес-стратегию, идеи и так далее. Вот. Поэтому, наверное, в этом плане все хорошо. То есть я особо лично не сталкивалась со всей этой тематикой, поэтому мне сложно как-то, да, от себя говорить. Вот. Но вот я вижу, что ребята вот, да, с которыми я общаюсь, кто занимается бизнесом, туго, медленно, но идет. Но они очень много работают. То есть у меня парень работает, вот вчера он пришел домой в 11 вечера. Вот. Ушел из дома в 8 утра. Весь день вот как бы он на работе. Пришел такой: «Я голодный». Я говорю: «Что ты ел за день?» – «Ну, булочку скушал на завтрак». Все, больше ничего не ел, потому что у него не было времени. Ему надо работать, чтобы его бизнес развивался. У него просто, ну, нет других вариантов. Вот. Опять-таки, не знаю, можно ли это сказать про там конкретно про нашу страну, мне кажется это просто как бы специфика открытия своего дела. Вот. Где бы ты не жил и что чем бы ты не занимался, да, в своем бизнесе. Вот. То есть можно быть вот таким лайтовым самозанятым, как я планирую, да, но опять-таки не делать какого-то настолько большого дела и не зарабатывать настолько больших денег, как планируют ребята. Вот. А если хочется больших денег, то нужно, ну действительно, во-первых, приложить очень много времени и усилий, и, во-вторых, ожидать, что как все равно может ничего не получиться. Вот. Как-то так.
Интервьюер: Угу. Ну, в целом у меня как бы по содержательным все, но в конце мы обычно всегда задаем вопрос а-ля про светлое будущее, типа там твои ближайшие планы именно там, может, тебе чему-то хочется научиться, куда-то съездить, что-то сделать.
Информант: Вообще я очень надеюсь, что у нас с молодым человеком все вопросы будут разрешены. Вот. Потому что по сути-то остались они только у него [смех] Он, ну, он постарше меня, он уже хочет все – и семью, и детей. Как бы я в целом тоже уже достаточно готова. То есть у меня, наверное, планы такие достаточно типичные. Закончить все-таки наконец-то университет [смех] Дай Боже, оно случится. Вот. Возможно. Вообще у меня тема диплома была подготовлена. Тема диплома очень классная. На эту тему еще никто не писал, никаких работ в мире по ней нет. Мы должны были с моим научруком над ней работать. Научрук у меня вообще безумно крутой мужчина. Он молодой, во-первых, ему 35 лет. Он слепой. Он, будучи слепым, изучил физику, изучил математику. То есть для меня этот человек – это просто бесконечное удивление, вдохновение. Типа насколько человеку было действительно интересно и хотелось этим заниматься. Вот. Очень умный, очень умный мужчина. Вот. Он мне дал сложную тему, но при этом достаточно интересную. Там про черные дыры. Вот. Так если вкратце там про определенный процесс, который происходит на границе черной дыры. Вот. И в общем-то если все-таки я напишу на эту тему, да, и никто не напишет раньше меня, то это можно будет куда-то продвигать, возможно, меня куда-то возьмут, возможно, я смогу развиваться в этой сфере. Параллельно все равно собираюсь заниматься фотографией, потому что, во-первых, мне это нравится, во-вторых, я не надеюсь, что физика сможет принести мне деньги. Потому что пока что все-таки я не вижу себя там переезжающей куда-то за границу, потому что, во-первых, очень сложно, очень дорого, и, ну, пока что мне кажется, что игра не стоит свеч в плане переезда. Может быть, когда-нибудь получится, не знаю. Вот. А в России я на физике много не заработаю, поэтому какой-то источник дохода нужен. Почему бы этому не быть фотографии, если она мне нравится, она может приносить достаточно денег, в общем-то все прекрасно. То есть, наверное, как-то так. Вот в этих, среди этих двух сфер как-то разобраться, найти себя, найти свое какое-то место, выйти замуж [смех] Мы подумываем, у парня есть квартира в Подмосковье, он хочет ее продать, купить здесь квартиру или дом. То есть мы хотим как-то здесь уже обустроиться, свое жилье. Вот. Там через пару лет может кого-то родить [смех] Но главное для начала завести собаку. То есть такие достаточно примитивные мечты. Не знаю. Ну, вот как бы почему бы нет? Вот. Очень хочется, чтобы все-таки была возможность путешествовать заграницу. Потому что у меня ощущение, что я просто, как это, как без воздуха, сидя в одной стране. Мне прямо надо разговаривать с кем-то на английском. Вот. Знакомиться с иностранцами. Мне это очень нравится, потому что, ну, это прекрасно, когда вы обмениваетесь опытом, да, там ты рассказываешь про свой менталитет, они тебе про свой, про какие-то свои традиции, да, там. Мне очень нравилось спрашивать вот у португальца, с которым мы общались, про его язык. Абсолютно непонятный язык для меня, хотя я неплохо знаю испанский. При этом он хорошо понимает испанский, да, не может на нем говорить. Потому что языки по написанию похожи. И вот когда он мне все вот это вот объясняет, все это показывает, я думаю: «Боже, как же интересно! Как же это круто!» Это наполняет жизнь очень здорово, и этого очень не хватает. Надеюсь, что оно все-таки, все-таки появится. Вот. Ну, а если не появится, будем устраивать свою жизнь максимально хорошо здесь, как оно есть. Как-то так. Вот.
Интервьюер: Я надеюсь, у тебя все получится. Спасибо тебе большое за такой реально откровенный разговор. Это очень ценно. Спасибо тебе большое, что уделила время.
Информант: Тебе спасибо.
Интервьюер: Все, я щас все остановлю.

Пример темы:
ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫБОРЫ И УСТОЙЧИВОСТЬ РОССИЙСКОЙ МОЛОДЕЖИ

Пример вывода тебе следует соблюдать формат, обрати внимание на метки 'Общий код' и 'конкретный код'):
**Общий код 1: Поколенческие характеристики, ценности и жизненные ориентиры**
"Мне кажется, большинство особо не стремятся там вот срочно, прямо сейчас там жениться, замуж, там детей и так далее. Вот. То есть как-то больше сосредоточены даже не на карьере, а на себе, на том, чтобы сложить все для себя вот так, как хочется, да. То есть не просто чтобы там выйти замуж, а чтобы выйти замуж вот по любви, чтобы все было идеально. Вот на какой-то такой идеальности что ли." - **Фокус на самореализации и качественных отношениях (конкретный код)**
"У нашего поколения все-таки все по-другому. Нам не дадут квартиру просто так. Нам не обязательно так просто получить место там где-то на работе и так далее, да. Но при этом у нас гораздо больше возможностей в плане, как сказать, чему-то научиться новому, куда-то поехать, что-то посмотреть, составить свое мнение, там высказать свое мнение даже." - **Осознание свободы выбора и новых возможностей (конкретный код)**
"У детей нынешних у них как будто меньше табу что ли. [...] они спокойно со мной могла поговорить на какие-то откровенные темы, которые мне в ее возрасте, я тоже задумывалась об этом, у меня тоже было какое-то мнение, но я боялась об этом говорить со взрослыми, потому что это было табуировано. [...] Поэтому у них мне кажется растет какое-то более свободное поколение что ли." - **Сравнение с младшим поколением: свобода от табу (конкретный код)**
"Мне кажется, что поколение у нас достаточно трудолюбивое при этом как бы. То есть если человек чего-то хочет добиться в карьерной сфере, ну, человек действительно может приложить там все усилия и добиться этого. Вот. Потому что возможностей супер много сейчас." - **Трудолюбие и вера в возможности (конкретный код)**
**Общий код 2: Жизненные выборы (образование, карьера, переезд)**
"Я хотела поступать в какой-нибудь другой город... Но мама сразу резко сказала: "Нет, на это у нас нет денег, я не смогу тебя содержать где-то там". [...] Поэтому пришлось все-таки остановиться на Питере." - **Выбор между амбициями и семейными обстоятельствами (конкретный код)**
"В [первый институт] я не закончила... мне был 21 год, оно было как раз к месту, вовремя, и университет гораздо, как сказать, дружелюбно настроенный к студентам нежели в [первый институт] это было. [...] Я сейчас в принципе университетом очень довольна." - **Осознанный выбор вуза и среды после неудачного опыта (конкретный код)**
"Сейчас для меня это очень ценно, потому что, ну, в связи с тем, почему я взяла академ, как бы у меня есть сложности иногда с выходом из дома. [...] Мне комфортно если работать с людьми, то один на один. Либо вообще работать не с людьми, а что-то другое делать. Поэтому я вот сейчас когда я снимаю, мне вообще вполне себе прекрасно." - **Выбор фриланса и фотографии из-за личных обстоятельств (конкретный код)**
"Мысли уехать были, но парень у меня все равно как-то прям ни в какую не хотел. То есть он такой очень ярый патриот своей страны. Он никуда не хочет уезжать. Вот. А я как-то наоборот, мне всегда было интересно пожить где-то в другом месте." - **Конфликт ценностей в паре: эмиграция vs патриотизм (конкретный код)**
**Общий код 3: Устойчивость и стратегии преодоления трудностей**
"Когда бабушки не стало, я поступила в университет, я параллельно там работала, я параллельно там, не знаю, танцами занималась, еще там чем-то занималась. То есть у меня все расписание было просто минута в минуту [...] И это позволяло мне просто, ну, не думать, не переживать о том, что вот бабушки не стало." - **Стратегия преодоления горя через сверхзанятость (конкретный код)**
"Мне нужно было время просто наедине с собой, чтобы меня никто не трогал и ничего не заботило. Вот. Ну, то есть, да, наверное, побыть один на один с собой." - **Необходимость уединения как способ справиться с трудностями (конкретный код)**
"У меня всегда, всегда, что бы ни произошло, даже мелочь какая-то, мне сразу надо кому-то позвонить и рассказать. То есть либо там подружке, либо маме, но вот это выговориться — это обязательно, это всегда помогает." - **Выговориться как ключевая копинг-стратегия (конкретный код)**
"Если бы не антидепрессанты, че бы я делала. [...] Плюс, наверное, очень хорошо, что у меня сейчас была возможность, то есть дать себе, собственно говоря, посидеть дома. То есть не работать, не учиться какое-то время. Вот. Это вот большое спасибо моему парню за это." - **Принятие помощи (медицинской и от партнера) как ресурс (конкретный код)**
**Общий код 4: Влияние макрособытий (СВО, корона) на идентичность и жизнь**
"Когда началась [специальная военная операция], задело меня очень сильно. Потому что, как я уже упоминала, у меня корни с Украины. [...] У меня в детстве бабушка со мной на украинском разговаривала. [...] И как-то мне бабушка прививала любовь к этой культуре, именно что вот всегда говорила: "Ты украинка". [...] И тут когда начинается [СВО] между как бы странами... это тяжело ударяет." - **Внутриличностный конфликт из-за двойной идентичности (конкретный код)**
"Очень многие уехали. Однозначно. [...] Есть немножечко ощущение, что общество как бы разделилось на тех, кто за, и тех, кто против. [...] Многим людям, в том числе и мне, которые до этого были такие аполитичные... пришлось так или иначе хоть какое-то мнение о политике составить." - **Раскол в обществе и вынужденная политизация молодежи (конкретный код)**
"В какой-то степени свободы слова лишились. То есть как бы я не против нашей власти, но я против [СВО]. Но я не могу пойти открыто об этом сказать, потому что меня сразу в кутузку посадят. Это тоже страшно. Даже если не посадят в кутузку, банально я студент вуза, меня могут отчислить." - **Страх репрессий и самоцензура (конкретный код)**
"Если корона. Очень сложно я ее восприняла морально, потому что у меня прям все, все было четко распланировано в общем. В плане тех же путешествий. [...] И все эти планы рухнули, потому что ровно как раз в 20-х числах марта началась корона... Сейчас мне, наоборот, очень комфортно сидеть дома, чтобы меня лишний раз никто не трогал." - **Крушение планов и смена паттерна поведения (от экстраверсии к интроверсии) (конкретный код)**
**Общий код 5: Представления о будущем и надежда**
"Я очень надеюсь, что [СВО] закончится. На самом деле... не знаю, может, так плохо говорить, надеюсь, не в нашу пользу. Потому что очень жалко Украину. [...] С другой стороны, как бы тоже совсем, знаешь, жить в побежденной стране тоже не очень хочется, потому что очень, очень много чего плохого это за собой повлечет для нас." - **Амбивалентность в отношении исхода войны (конкретный код)**
"Очень хочется, чтобы все-таки была возможность путешествовать заграницу. Потому что у меня ощущение, что я просто, как это, как без воздуха, сидя в одной стране. Мне прямо надо разговаривать с кем-то на английском. Вот. Знакомиться с иностранцами." - **Тоска по открытому миру и межкультурному общению (конкретный код)**
"Я очень надеюсь, что у нас с молодым человеком все вопросы будут разрешены. [...] Закончить все-таки наконец-то университет. [...] Мы хотим как-то здесь уже обустроиться, свое жилье. Вот. Там через пару лет может кого-то родить. Но главное для начала завести собаку." - **Типичные жизненные планы в условиях нестабильности (конкретный код)**
"Ну, а если не появится [возможность путешествовать], будем устраивать свою жизнь максимально хорошо здесь, как оно есть. Как-то так." - **Адаптивный оптимизм и принятие реальности (конкретный код)**

Теперь выполни разметку для нового интервью. Важно: ответ должен содержать только коды и цитаты, без лишних слов и повторов.
Дай ответ на русском языке.
"""

    user_content = f"""Тема интервью:
{example['topic']}

Текст интервью:
{example['transcript']}

Пожалуйста, выполни разметку интервью в указанном формате."""

    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return prompt

In [ ]:
def prepare_dataset(hf_dataset, tokenizer):
    queries, targets = [], []
    for ex in hf_dataset:
        queries.append(build_prompt_h1(ex, tokenizer))
        targets.append(ex['coding'])
    return Dataset.from_dict({"query": queries, "coding": targets})

def extract_source_and_topic(input_text):
    """
    Извлекает транскрипт и тему из поля input_text.
    Формат: "Тема интервью: ...\nТекст интервью: ..." (транскрипт может быть многострочным)
    """
    topic = ""
    transcript = ""

    lines = input_text.split('\n')
    for line in lines:
        if line.startswith("Тема интервью:"):
            topic = line.replace("Тема интервью:", "").strip()
            break

    if not topic and "Тема интервью:" in input_text:
        first_part = input_text.split("Текст интервью:")[0]
        if "Тема интервью:" in first_part:
            topic = first_part.split("Тема интервью:", 1)[1].strip()

    marker = "Текст интервью:"
    if marker in input_text:
        transcript = input_text.split(marker, 1)[1].strip()
    else:
        found = False
        transcript_lines = []
        for line in lines:
            if line.startswith("Текст интервью:"):
                found = True
                transcript_lines.append(line.replace("Текст интервью:", "").strip())
            elif found:
                transcript_lines.append(line)
        transcript = '\n'.join(transcript_lines).strip()

    return transcript, topic

Подготовка тестового датасета

In [ ]:
test_dataset = prepare_dataset(test_dataset, tokenizer)
test_source_texts = []
test_topics = []
for ex in test_dataset:
    src, top = extract_source_and_topic(ex["query"])
    test_source_texts.append(src)
    test_topics.append(top)
test_dataset = test_dataset.add_column("transcript", test_source_texts)
test_dataset = test_dataset.add_column("topic", test_topics)

Функции генерации

In [ ]:
def clear_cuda_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()

def generate_code(prompt, max_new_tokens=512, num_beams=3):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1800).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=num_beams,
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated

Инференс и сбор метрик

In [ ]:
device = next(model.parameters()).device
predictions = []
all_rewards_total = []
all_rewards_bert = []
all_rewards_llm = []

for idx in tqdm(range(len(test_dataset)), desc="Evaluating"):
    batch_item = test_dataset[idx]
    query = batch_item["query"]
    target = batch_item["coding"]
    src = batch_item["transcript"]
    topic = batch_item["topic"]

    pred = generate_code(query)

    total_reward, bert_reward, llm_reward = compute_reward(pred, target, src, topic, use_llm=USE_LLM_REWARD)

    predictions.append({
        "query": query,
        "prediction": pred,
        "target": target,
        "source_text": src,
        "topic": topic,
        "reward_total": total_reward,
        "reward_bert": bert_reward,
        "reward_llm": llm_reward
    })

pred_df = pd.DataFrame(predictions)

print("\n=== Test Metrics ===")

print(f'\nGeneration strategy: Beam Search')
print(f"Average Total Reward: {np.mean(pred_df['reward_total']):.4f}")
print(f"Average BERTScore: {np.mean(pred_df['reward_bert']):.4f}")
print(f"Average LLM Score: {np.mean(pred_df['reward_llm']):.4f}")


Evaluating: 100%|██████████| 23/23 [2:16:17<00:00, 355.55s/it]


=== Test Metrics ===

Generation strategy: Beam Search
Average Total Reward: 0.5588
Average BERTScore: 0.5551
Average LLM Score: 0.0036


In [ ]:
pred_df.to_csv('/content/drive/MyDrive/psad_project/Evaluation/qwen7b_prompt_engineering/7b_base_h1_predictions_with_scores.csv', index=False)
print("Результаты сохранены в 7b_base_h1_predictions_with_scores.csv")
pred_df.head()

Результаты сохранены в 7b_base_h1_predictions_with_scores.csv


,query,prediction,target,source_text,topic,reward_total,reward_bert,reward_llm
0,<|im_start|>system\nТы — эксперт по анализу ин...,. Вот.\nИнтервьюер: То есть ты поступила в [пе...,**Общий код 1: Переход к самостоятельности: ма...,"Интервьюер: Хорошо. Вот, согласны ли вы на зап...","Тема: МНОГООБРАЗИЕ ФОРМ, ЭФФЕКТОВ И БАРЬЕРОВ И...",0.547806,0.547806,0.0
1,<|im_start|>system\nТы — эксперт по анализу ин...,. Вот.\nИнтервьюер: То есть ты поступила в [пе...,"**Общий код 1: Поколенческие характеристики, ц...",Интервьюер: Как всегда начнем со знакомства. Р...,"Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...",0.551183,0.551183,0.0
2,<|im_start|>system\nТы — эксперт по анализу ин...,. Вот.\nИнтервьюер: То есть ты поступила в [пе...,**Общий код 1: Представления о работе и критер...,"Интервьюер: Начинаю. Первый вопрос расскажи, п...",Тема: ПОКОЛЕНИЕ Z В ПОИСКАХ БАЛАНСА: УСЛОВИЯ Т...,0.545178,0.545178,0.0
3,<|im_start|>system\nТы — эксперт по анализу ин...,. Вот.\nИнтервьюер: То есть ты поступила в [пе...,**Общий код 1: Переход к самостоятельности: пе...,"Интервьюер: Согласие на запись даешь, верно?\n...","Тема: МНОГООБРАЗИЕ ФОРМ, ЭФФЕКТОВ И БАРЬЕРОВ И...",0.569308,0.569308,0.0
4,<|im_start|>system\nТы — эксперт по анализу ин...,. Вот.\nИнтервьюер: То есть ты поступила в [пе...,**Общий код 1: Переход к самостоятельности: вз...,"Интервьюер: Согласие ты даешь, верно?\nИнформа...","Тема: МНОГООБРАЗИЕ ФОРМ, ЭФФЕКТОВ И БАРЬЕРОВ И...",0.538333,0.538333,0.0
